In [1]:
import json
import pandas as pd
import json
from collections import Counter

In [2]:
case_df = pd.read_parquet(
    "../data/case_7_data_for_rd.snappy.parquet"
)

truth_df = pd.read_parquet(
    "../data/2_truth_rd_data.snappy.parquet"
)

In [3]:
truth_unique = truth_df.drop_duplicates().copy()

In [4]:
base_truth = (
    truth_unique
    .groupby("rd_number")
    .agg(
        tg=("tg", lambda x: sorted(set(x))),
        rd_type=("rd_type", lambda x: sorted(set(x))),
        tnved_codes=(
            "code_tnved",
            lambda x: sorted(set(x.dropna()))
        ),
    )
    .reset_index()
)

In [5]:
matched_numbers = set(base_truth["rd_number"])

In [6]:
mask = (
    case_df["rank"].eq(1) &
    case_df["rd_documentnumber"].isin(matched_numbers)
)

base_rd = case_df.loc[
    mask,
    ["rd_documentnumber", "rd_data"]
].copy()

In [7]:
base_df = base_rd.merge(
    base_truth,
    left_on="rd_documentnumber",
    right_on="rd_number",
    how="inner"
)

In [8]:
print(base_df.shape)
print(base_df["rd_documentnumber"].nunique())
print(base_df["rd_documentnumber"].duplicated().sum())

(86301, 6)
86301
0


In [9]:
applicant_keys = Counter()
applicant_types = Counter()
applicant_nonempty = Counter()

In [10]:
for row in base_df["rd_data"]:
    data = json.loads(row)

    applicant = data.get("applicant")

    applicant_types[type(applicant).__name__] += 1

    if isinstance(applicant, dict):
        applicant_keys.update(applicant.keys())

        for key, value in applicant.items():
            if value not in ("", None, [], {}):
                applicant_nonempty[key] += 1

print("Типы applicant:")
print(applicant_types)

print("\nКлючи applicant:")
print(applicant_keys.most_common())

print("\nНепустые значения:")
print(applicant_nonempty.most_common())

Типы applicant:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи applicant:
[('address', 76632), ('fullName', 76298), ('type', 75017), ('ogrn', 74989), ('email', 74950), ('inn', 74835), ('phone', 74674), ('directorName', 74636)]

Непустые значения:
[('address', 74799), ('fullName', 74469), ('type', 72775), ('ogrn', 72480), ('directorName', 72339), ('inn', 72183), ('email', 72093), ('phone', 71890)]


In [11]:
applicant_stats = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        for key in applicant:
            applicant_stats[f"{key}__field_present"] += 1

            value = applicant.get(key)

            if value not in ("", None, [], {}):
                applicant_stats[f"{key}__nonempty"] += 1

applicant_stats

Counter({'address__field_present': 76632,
         'fullName__field_present': 76298,
         'type__field_present': 75017,
         'ogrn__field_present': 74989,
         'email__field_present': 74950,
         'inn__field_present': 74835,
         'address__nonempty': 74799,
         'phone__field_present': 74674,
         'directorName__field_present': 74636,
         'fullName__nonempty': 74469,
         'type__nonempty': 72775,
         'ogrn__nonempty': 72480,
         'directorName__nonempty': 72339,
         'inn__nonempty': 72183,
         'email__nonempty': 72093,
         'phone__nonempty': 71890})

In [12]:
def has_nonempty_dict(value):
    return isinstance(value, dict) and any(
        v not in ("", None, [], {})
        for v in value.values()
    )

In [13]:
base_df["applicant_present"] = (
    base_df["rd_data"]
    .map(lambda x: has_nonempty_dict(json.loads(x).get("applicant")))
)

In [14]:
base_df["applicant_present"].value_counts(normalize=True)

applicant_present
True     0.86824
False    0.13176
Name: proportion, dtype: float64

In [15]:
tg_exploded = base_df.explode("tg").copy()

In [16]:
tg_exploded.groupby("tg")["applicant_present"].agg(
    ["count", "mean"]
)

,count,mean
tg,,
35,68370,0.835279
4,3936,1.000000
43,14061,0.991039


In [17]:
applicant_types_values = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        value = applicant.get("type")

        if value not in ("", None):
            applicant_types_values[value] += 1

applicant_types_values.most_common(20)

[('Юридическое лицо', 65746), ('Индивидуальный предприниматель', 7029)]

In [18]:
applicant_records = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        applicant_records.append({
            "applicant_present": any(
                value not in ("", None, [], {})
                for value in applicant.values()
            ),
            "applicant_type": applicant.get("type") or None,
            "applicant_has_inn": bool(applicant.get("inn")),
            "applicant_has_ogrn": bool(applicant.get("ogrn")),
            "applicant_has_address": bool(applicant.get("address")),
            "applicant_has_email": bool(applicant.get("email")),
            "applicant_has_phone": bool(applicant.get("phone")),
            "applicant_has_director": bool(applicant.get("directorName")),
            "applicant_has_fullname": bool(applicant.get("fullName")),
        })
    else:
        applicant_records.append({
            "applicant_present": False,
            "applicant_type": None,
            "applicant_has_inn": False,
            "applicant_has_ogrn": False,
            "applicant_has_address": False,
            "applicant_has_email": False,
            "applicant_has_phone": False,
            "applicant_has_director": False,
            "applicant_has_fullname": False,
        })

applicant_features = pd.DataFrame(
    applicant_records,
    index=base_df.index
)

In [19]:
base_df = pd.concat(
    [base_df, applicant_features],
    axis=1
)

In [20]:
base_df = base_df.loc[:, ~base_df.columns.duplicated()].copy()
base_df.columns[base_df.columns.duplicated()].tolist()

[]

In [21]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
)

In [22]:
print(base_df.index.is_unique)
print(base_df.index[:10])

True
RangeIndex(start=0, stop=10, step=1)


In [23]:
type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["applicant_type"],
    normalize="index"
)

type_by_tg

applicant_type,Индивидуальный предприниматель,Юридическое лицо
tg,,
35,0.102638,0.897362
4,0.224942,0.775058
43,0.034972,0.965028


In [24]:
doc_tg_type = (
    truth_unique[
        ["rd_number", "tg", "rd_type"]
    ]
    .loc[
        lambda df: df["rd_number"].isin(
            base_df["rd_documentnumber"]
        )
    ]
    .drop_duplicates()
    .copy()
)

In [25]:
print(doc_tg_type.shape)
doc_tg_type.head()

(86371, 3)


,rd_number,tg,rd_type
0,ЕАЭС N RU Д-IT.РА03.В.08011/25,4,N/A
1,ЕАЭС N RU Д-FR.РА01.В.63599/25,4,N/A
2,ЕАЭС N RU Д-FR.РА08.В.83499/24,4,N/A
3,ЕАЭС N RU Д-ES.РА02.В.52976/25,4,N/A
4,ЕАЭС N RU Д-ES.РА09.В.91241/23,4,N/A


In [26]:
applicant_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "applicant_present",
            "applicant_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [27]:
print(applicant_tg_type.shape)
print(applicant_tg_type["rd_number"].nunique())

(86371, 5)
86301


In [28]:
applicant_by_tg_type = (
    applicant_tg_type
    .groupby(["tg", "rd_type"])["applicant_present"]
    .agg(
        count="count",
        mean="mean"
    )
)

applicant_by_tg_type

count  mean
tg rd_type             
35 ДС       56841   1.0
   СГР      11262   0.0
   СС         271   1.0
4  N/A       3936   1.0
43 ДС       13913   1.0
   СГР        126   0.0
   СС          22   1.0

In [29]:
type_by_tg_type = pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"],
    normalize="index"
)

type_by_tg_type

applicant_type  Индивидуальный предприниматель  Юридическое лицо
tg rd_type                                                      
35 ДС                                 0.103115          0.896885
   СС                                 0.007407          0.992593
4  N/A                                0.224942          0.775058
43 ДС                                 0.035029          0.964971
   СС                                 0.000000          1.000000

In [30]:
applicant_type_stats = (
    applicant_tg_type
    .assign(
        applicant_type_clean=
        applicant_tg_type["applicant_type"].fillna("Нет данных")
    )
    .groupby(["tg", "rd_type"])["applicant_type_clean"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
)

applicant_type_stats

,tg,rd_type,applicant_type_clean,share
0,35,ДС,Юридическое лицо,0.870235
1,35,ДС,Индивидуальный предприниматель,0.100051
2,35,ДС,Нет данных,0.029714
3,35,СГР,Нет данных,1.000000
4,35,СС,Юридическое лицо,0.988930
5,35,СС,Индивидуальный предприниматель,0.007380
6,35,СС,Нет данных,0.003690
7,4,N/A,Юридическое лицо,0.764228
8,4,N/A,Индивидуальный предприниматель,0.221799
9,4,N/A,Нет данных,0.013974


In [31]:
pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"].fillna("Нет данных")
)

applicant_type  Индивидуальный предприниматель  Нет данных  Юридическое лицо
tg rd_type                                                                  
35 ДС                                     5687        1689             49465
   СГР                                       0       11262                 0
   СС                                        2           1               268
4  N/A                                     873          55              3008
43 ДС                                      473         410             13030
   СГР                                       0         126                 0
   СС                                        0           0                22

In [32]:
pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"].fillna("Нет данных")
)

applicant_type  Индивидуальный предприниматель  Нет данных  Юридическое лицо
tg rd_type                                                                  
35 ДС                                     5687        1689             49465
   СГР                                       0       11262                 0
   СС                                        2           1               268
4  N/A                                     873          55              3008
43 ДС                                      473         410             13030
   СГР                                       0         126                 0
   СС                                        0           0                22

In [33]:
tg_exploded["is_ip"] = (
    tg_exploded["applicant_type"]
    == "Индивидуальный предприниматель"
)

pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["is_ip"],
    normalize="index"
)

is_ip,False,True
tg,,
35,0.916806,0.083194
4,0.778201,0.221799
43,0.966361,0.033639


In [34]:
ip_docs = tg_exploded[
    tg_exploded["applicant_type"]
    == "Индивидуальный предприниматель"
]

ip_docs["tg"].value_counts(normalize=True)

tg
35    0.808644
4     0.124111
43    0.067245
Name: proportion, dtype: float64

In [35]:
ip_share_by_tg = (
    tg_exploded
    .groupby("tg")["is_ip"]
    .mean()
)

overall_ip_share = tg_exploded["is_ip"].mean()

ip_lift = ip_share_by_tg / overall_ip_share

ip_lift

tg
35    1.021503
4     2.723357
43    0.413038
Name: is_ip, dtype: float64

In [36]:
ip_share_by_tg = (
    tg_exploded
    .groupby("tg")["is_ip"]
    .mean()
)

overall_ip_share = tg_exploded["is_ip"].mean()

ip_lift = ip_share_by_tg / overall_ip_share

pd.DataFrame({
    "ip_share": ip_share_by_tg,
    "ip_lift": ip_lift
})

,ip_share,ip_lift
tg,,
35,0.083194,1.021503
4,0.221799,2.723357
43,0.033639,0.413038


In [37]:
manufacturer_types = Counter()
manufacturer_keys = Counter()
manufacturer_nonempty = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    manufacturer_types[type(manufacturer).__name__] += 1

    if isinstance(manufacturer, dict):
        manufacturer_keys.update(manufacturer.keys())

        for key, value in manufacturer.items():
            if value not in ("", None, [], {}):
                manufacturer_nonempty[key] += 1

print("Типы manufacturer:")
print(manufacturer_types)

print("\nКлючи manufacturer:")
print(manufacturer_keys.most_common())

print("\nНепустые значения:")
print(manufacturer_nonempty.most_common())

Типы manufacturer:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи manufacturer:
[('name', 76682), ('address', 76545), ('type', 75017), ('inn', 74378), ('filialAddresses', 66736), ('GLN', 22222)]

Непустые значения:
[('name', 74908), ('address', 74706), ('type', 72775), ('inn', 70034), ('filialAddresses', 60522)]


In [38]:
manufacturer_present_values = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    manufacturer_present_values.append(
        isinstance(manufacturer, dict) and any(
            value not in ("", None, [], {})
            for value in manufacturer.values()
        )
    )

base_df["manufacturer_present"] = manufacturer_present_values

In [39]:
pd.crosstab(
    base_df["applicant_present"],
    base_df["manufacturer_present"],
    normalize="all"
)

manufacturer_present,False,True
applicant_present,,
False,0.13176,0.00000
True,0.00000,0.86824


In [40]:
(
    base_df["applicant_present"]
    == base_df["manufacturer_present"]
).value_counts(normalize=True)

True    1.0
Name: proportion, dtype: float64

In [41]:
manufacturer_type = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    if isinstance(manufacturer, dict):
        manufacturer_type.append(
            manufacturer.get("type") or None
        )
    else:
        manufacturer_type.append(None)

base_df["manufacturer_type"] = manufacturer_type

In [42]:
base_df["manufacturer_type"].value_counts(dropna=False)

manufacturer_type
Иностранное юридическое лицо      33932
Юридическое лицо                  32568
NaN                               13526
Индивидуальный предприниматель     3927
Физическое лицо                    2348
Name: count, dtype: int64

In [43]:
tg_exploded = base_df.explode("tg", ignore_index=True)

In [44]:
manufacturer_type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

manufacturer_type_by_tg

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
tg,,,,,
35,0.048662,0.374345,0.189440,0.018926,0.368627
4,0.125508,0.670986,0.013974,0.011179,0.178354
43,0.007823,0.406941,0.038120,0.071972,0.475144


In [45]:
manufacturer_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "manufacturer_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [46]:
print(manufacturer_tg_type.shape)
print(manufacturer_tg_type["rd_number"].nunique())

(86371, 4)
86301


In [47]:
manufacturer_type_by_tg_type = pd.crosstab(
    [manufacturer_tg_type["tg"], manufacturer_tg_type["rd_type"]],
    manufacturer_tg_type["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

manufacturer_type_by_tg_type

manufacturer_type  Индивидуальный предприниматель  \
tg rd_type                                          
35 ДС                                    0.058532   
   СГР                                   0.000000   
   СС                                    0.003690   
4  N/A                                   0.125508   
43 ДС                                    0.007906   
   СГР                                   0.000000   
   СС                                    0.000000   

manufacturer_type  Иностранное юридическое лицо  Нет данных  Физическое лицо  \
tg rd_type                                                                     
35 ДС                                  0.450256    0.029714         0.022730   
   СГР                                 0.000000    1.000000         0.000000   
   СС                                  0.011070    0.003690         0.007380   
4  N/A                                 0.670986    0.013974         0.011179   
43 ДС                                  0.410120    0.029469         0.072450   
   СГР                                 0.000000    1.000000         0.000000   
   СС                                  0.727273    0.000000         0.181818   

manufacturer_type  Юридическое лицо  
tg rd_type                           
35 ДС                      0.438768  
   СГР                     0.000000  
   СС                      0.974170  
4  N/A                     0.178354  
43 ДС                      0.480055  
   СГР                     0.000000  
   СС                      0.090909

In [48]:
manufacturer_precision_view = pd.crosstab(
    [
        manufacturer_tg_type["rd_type"],
        manufacturer_tg_type["manufacturer_type"].fillna("Нет данных")
    ],
    manufacturer_tg_type["tg"],
    normalize="index"
)

manufacturer_precision_view

tg                                            35    4        43
rd_type manufacturer_type                                      
N/A     Индивидуальный предприниматель  0.000000  1.0  0.000000
        Иностранное юридическое лицо    0.000000  1.0  0.000000
        Нет данных                      0.000000  1.0  0.000000
        Физическое лицо                 0.000000  1.0  0.000000
        Юридическое лицо                0.000000  1.0  0.000000
ДС      Индивидуальный предприниматель  0.967995  0.0  0.032005
        Иностранное юридическое лицо    0.817694  0.0  0.182306
        Нет данных                      0.804669  0.0  0.195331
        Физическое лицо                 0.561739  0.0  0.438261
        Юридическое лицо                0.788766  0.0  0.211234
СГР     Нет данных                      0.988936  0.0  0.011064
СС      Индивидуальный предприниматель  1.000000  0.0  0.000000
        Иностранное юридическое лицо    0.157895  0.0  0.842105
        Нет данных                      1.000000  0.0  0.000000
        Физическое лицо                 0.333333  0.0  0.666667
        Юридическое лицо                0.992481  0.0  0.007519

In [49]:
app_manufacturer = pd.crosstab(
    base_df["applicant_type"].fillna("Нет данных"),
    base_df["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

app_manufacturer

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
applicant_type,,,,,
Индивидуальный предприниматель,0.542182,0.354531,0.0,0.018495,0.084792
Нет данных,0.000000,0.000000,1.0,0.000000,0.000000
Юридическое лицо,0.001764,0.478204,0.0,0.033736,0.486296


In [50]:
product_types = Counter()
product_keys = Counter()
product_nonempty = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    product_types[type(product).__name__] += 1

    if isinstance(product, dict):
        product_keys.update(product.keys())

        for key, value in product.items():
            if value not in ("", None, [], {}):
                product_nonempty[key] += 1

print("Типы product:")
print(product_types)

print("\nКлючи product:")
print(product_keys.most_common())

print("\nНепустые значения:")
print(product_nonempty.most_common())

Типы product:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи product:
[('idObjectType', 76684), ('identification', 76684), ('productName', 76675), ('tnved', 76618), ('productInfo', 73283), ('idProductOrigin', 69738)]

Непустые значения:
[('idObjectType', 74930), ('productName', 74863), ('tnved', 74852), ('productInfo', 68544), ('identification', 68425), ('idProductOrigin', 64755)]


In [51]:
product_object_type = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    if isinstance(product, dict):
        product_object_type.append(
            product.get("idObjectType") or None
        )
    else:
        product_object_type.append(None)

base_df["product_object_type"] = product_object_type

In [52]:
base_df["product_object_type"].value_counts(dropna=False)

product_object_type
Серийный выпуск      73437
NaN                  11371
Партия                1482
Единичное изделие       11
Name: count, dtype: int64

In [53]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
).copy()

In [54]:
product_object_type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["product_object_type"].fillna("Нет данных"),
    normalize="index"
)

product_object_type_by_tg

product_object_type,Единичное изделие,Нет данных,Партия,Серийный выпуск
tg,,,,
35,0.000102,0.164721,0.011452,0.823724
4,0.000254,0.000000,0.023882,0.975864
43,0.000213,0.008961,0.043027,0.947799


In [55]:
product_features = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    if isinstance(product, dict):
        product_features.append({
            "product_object_type": product.get("idObjectType") or None,
            "product_tnved": product.get("tnved") or None,
            "product_origin": product.get("idProductOrigin") or None,
            "product_info": product.get("productInfo") or None,
            "product_name": product.get("productName") or None,
            "product_identification": product.get("identification"),
        })
    else:
        product_features.append({
            "product_object_type": None,
            "product_tnved": None,
            "product_origin": None,
            "product_info": None,
            "product_name": None,
            "product_identification": None,
        })

product_features = pd.DataFrame(
    product_features,
    index=base_df.index
)

base_df[product_features.columns] = product_features

In [56]:
product_object_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "product_object_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [57]:
product_object_type_by_tg_type = pd.crosstab(
    [product_object_tg_type["tg"], product_object_tg_type["rd_type"]],
    product_object_tg_type["product_object_type"].fillna("Нет данных"),
    normalize="index"
)

product_object_type_by_tg_type

product_object_type  Единичное изделие  Нет данных    Партия  Серийный выпуск
tg rd_type                                                                   
35 ДС                         0.000123         0.0  0.013775         0.986102
   СГР                        0.000000         1.0  0.000000         0.000000
   СС                         0.000000         0.0  0.000000         1.000000
4  N/A                        0.000254         0.0  0.023882         0.975864
43 ДС                         0.000216         0.0  0.043485         0.956300
   СГР                        0.000000         1.0  0.000000         0.000000
   СС                         0.000000         0.0  0.000000         1.000000

In [58]:
base_df["product_tnved"].value_counts(dropna=False).head(30)

product_tnved
3304990000                22961
NaN                       11449
3305900009                 7417
3401300000                 5814
2710198200                 3744
3305100000                 3677
3303009000                 2765
2710198800                 1759
3304200000                 1655
3304100000                 1639
3403199000                 1578
3403990000                 1313
3402500000                 1248
3304300000                 1195
3307200000                 1141
3820000000                 1113
3401209000                  896
3307300000                  846
3401110001                  839
3307100000                  752
3808948000                  681
3306100000                  601
3303001000                  572
3304                        473
3304910000                  472
3819000000                  428
3306900000                  380
2710199800                  349
3403191000                  325
2710198200, 3403199000      321
Name: count, dtype: int64

In [59]:
base_df["product_tnved"].dropna().sample(
    30,
    random_state=42
).tolist()

['3304990000',
 '3401300000',
 '3305100000',
 '3403199000, 3403990000',
 '2710198200',
 '3401300000',
 '3304990000',
 '2710, 3403',
 '3305900009',
 '3820000000',
 '3304990000',
 '2710199200, 2710199800, 3403199000, 3403990000',
 '3403',
 '3305900009',
 '3304990000',
 '3307200000',
 '3304990000',
 '3304990000',
 '3401300000',
 '3808948000',
 '3401110001',
 '3401209000',
 '3304990000',
 '3304990000',
 '3304990000',
 '3403990000',
 '3401300000',
 '3403199000',
 '3401209000',
 '3304990000']

In [60]:
def split_tnved(value):
    if pd.isna(value) or not str(value).strip():
        return []

    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

In [61]:
base_df["tnved_list"] = base_df["product_tnved"].apply(split_tnved)

In [62]:
base_df["tnved_list"].head(20)

0     []
1     []
2     []
3     []
4     []
5     []
6     []
7     []
8     []
9     []
10    []
11    []
12    []
13    []
14    []
15    []
16    []
17    []
18    []
19    []
Name: tnved_list, dtype: object

In [63]:
base_df["tnved_list"].apply(len).value_counts().sort_index()

tnved_list
0      11449
1      71381
2       2211
3        687
4        287
5        115
6         55
7         45
8         19
9         16
10         3
11         2
13         2
15         2
16         2
19         1
20         1
23         2
25         2
26         2
32         1
78         1
107        1
201        1
213        1
216        1
220        1
228        1
229        2
232        2
237        1
243        1
258        1
276        1
279        1
Name: count, dtype: int64

In [64]:
tnved_lengths = (
    base_df["tnved_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.len()
)

tnved_lengths.value_counts().sort_index()

tnved_list
2        19
4      2653
6       582
9        73
10    81038
Name: count, dtype: int64

In [65]:
base_df["tnved_list"].explode().dropna().sample(
    30,
    random_state=42
).tolist()

['1704',
 '3403199000',
 '3304990000',
 '3401300000',
 '3304990000',
 '3307100000',
 '3403199000',
 '3305900009',
 '3307100000',
 '3305900009',
 '3305900009',
 '3401110009',
 '3304100000',
 '3403',
 '3305900009',
 '2710199800',
 '3306900000',
 '3305100000',
 '3304990000',
 '3304990000',
 '3401300000',
 '3403990000',
 '3304990000',
 '3305900000',
 '3304990000',
 '3305900009',
 '3304990000',
 '3303001000',
 '3304990000',
 '3305100000']

In [66]:
tnved_count = base_df["tnved_list"].apply(len)

tnved_count[tnved_count > 20].sort_values(ascending=False).head(20)

82407    279
82408    276
82415    258
82414    243
82413    237
82673    232
82411    232
82684    229
82404    229
82674    228
82409    220
82681    216
82412    213
82403    201
82682    107
82683     78
63366     32
74075     26
82406     26
57808     25
Name: tnved_list, dtype: int64

In [67]:
large_tnved_docs = tnved_count[tnved_count > 20].index[:5]

base_df.loc[
    large_tnved_docs,
    ["rd_documentnumber", "product_tnved", "tnved_list"]
]

,rd_documentnumber,product_tnved,tnved_list
51862,ЕАЭС N RU Д-RU.РА02.В.25158/25,"0907100000, 0907200000, 0908110000, 0908120000...","[0907100000, 0907200000, 0908110000, 090812000..."
57807,ЕАЭС N RU Д-RU.РА03.В.87862/24,"1521100000, 3301121000, 3301129000, 3301131000...","[1521100000, 3301121000, 3301129000, 330113100..."
57808,ЕАЭС N RU Д-RU.РА03.В.87863/24,"1521100000, 3301121000, 3301129000, 3301131000...","[1521100000, 3301121000, 3301129000, 330113100..."
63366,ЕАЭС N RU Д-RU.РА05.В.37375/23,"6104, 6104130000, 6104192000, 6104199001, 6104...","[6104, 6104130000, 6104192000, 6104199001, 610..."
69531,ЕАЭС N RU Д-RU.РА07.В.42910/24,"3301121000, 3301129000, 3301131000, 3301139000...","[3301121000, 3301129000, 3301131000, 330113900..."


In [68]:
print(
    (base_df["tnved_list"].apply(len) > 0).mean()
)

0.8673364155687652


In [69]:
tnved_codes = (
    base_df["tnved_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

tnved_codes.value_counts().head(30)

tnved_list
3304990000    23177
3305900009     7563
3401300000     5936
2710198200     4807
3305100000     3774
3403199000     3148
3303009000     3035
2710198800     2498
3403990000     2440
3304200000     1702
3304100000     1680
3402500000     1331
3304300000     1211
3820000000     1162
3307200000     1152
3401209000      945
3401110001      883
3307300000      871
3303001000      855
2710199800      825
3403191000      796
3307100000      765
3808948000      686
3306100000      603
2710198400      574
3304            555
3403            521
3819000000      515
3304910000      497
3306900000      381
Name: count, dtype: int64

In [70]:
def clean_tnved_list(codes):
    result = []

    for code in codes:
        code = str(code).strip()

        if code.isdigit():
            result.append(code)

    return result


base_df["tnved_list_clean"] = (
    base_df["tnved_list"].apply(clean_tnved_list)
)

In [71]:
def tnved_prefixes(codes, length):
    return sorted({
        code[:length]
        for code in codes
        if len(code) >= length
    })

In [72]:
base_df["tnved_2"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 2)
)

base_df["tnved_4"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 4)
)

base_df["tnved_6"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 6)
)

In [73]:
tnved_4_long = (
    base_df[
        ["rd_documentnumber", "tg", "tnved_4"]
    ]
    .explode("tnved_4", ignore_index=True)
    .dropna(subset=["tnved_4"])
)

In [74]:
tnved_4_long = tnved_4_long.drop_duplicates(
    subset=["rd_documentnumber", "tnved_4"]
)

In [75]:
tnved_4_long.groupby("tnved_4")["rd_documentnumber"].nunique()

tnved_4
0201     1
0202     3
0203     2
0204     1
0207     2
        ..
9403     1
9404     2
9503     1
9602     1
9603    19
Name: rd_documentnumber, Length: 258, dtype: int64

In [76]:
tnved_4_tg = (
    tnved_4_long
    .explode("tg")
    .groupby(["tnved_4", "tg"])["rd_documentnumber"]
    .nunique()
    .reset_index(name="docs")
)

In [77]:
tg_docs = (
    base_df
    .explode("tg", ignore_index=True)
    .groupby("tg")["rd_documentnumber"]
    .nunique()
)

tnved_4_tg["share"] = (
    tnved_4_tg["docs"]
    / tnved_4_tg["tg"].map(tg_docs)
)

In [78]:
tnved_4_tg.sort_values(
    ["tg", "share"],
    ascending=[True, False]
).groupby("tg").head(20)

,tnved_4,tg,docs,share
113,3304,35,28605,0.418385
116,3305,35,12317,0.180152
124,3401,35,8469,0.123870
121,3307,35,3294,0.048179
127,3402,35,1973,0.028858
138,3808,35,1055,0.015431
119,3306,35,990,0.014480
291,8212,35,72,0.001053
68,1905,35,60,0.000878
105,3301,35,44,0.000644


In [79]:
tnved_4_tg.sort_values(
    "share",
    ascending=False
).head(50)

,tnved_4,tg,docs,share
111,3303,4,3818,0.970020
94,2710,43,8447,0.600740
113,3304,35,28605,0.418385
131,3403,43,5701,0.405448
116,3305,35,12317,0.180152
124,3401,35,8469,0.123870
153,3820,43,1161,0.082569
121,3307,35,3294,0.048179
151,3819,43,514,0.036555
127,3402,35,1973,0.028858


In [80]:
tnved_4_tg_long = (
    tnved_4_long
    .explode("tg", ignore_index=True)
)

In [81]:
tnved_4_precision_view = pd.crosstab(
    tnved_4_tg_long["tnved_4"],
    tnved_4_tg_long["tg"],
    normalize="index"
)

In [82]:
tnved_4_precision_view

tg,35,4,43
tnved_4,,,
0201,1.0,0.0,0.0
0202,1.0,0.0,0.0
0203,1.0,0.0,0.0
0204,1.0,0.0,0.0
0207,1.0,0.0,0.0
...,...,...,...
9403,1.0,0.0,0.0
9404,1.0,0.0,0.0
9503,1.0,0.0,0.0


In [83]:
tnved_4_precision_view.loc[
    tnved_4_precision_view.max(axis=1).sort_values(ascending=False).head(30).index
]

tg,35,4,43
tnved_4,,,
9602,1.0,0.0,0.0
8705,1.0,0.0,0.0
8509,1.0,0.0,0.0
1003,1.0,0.0,0.0
0910,1.0,0.0,0.0
0909,1.0,0.0,0.0
0908,1.0,0.0,0.0
0907,1.0,0.0,0.0
0904,1.0,0.0,0.0


In [84]:
tnved_4_support = (
    tnved_4_tg_long
    .groupby("tnved_4")["rd_documentnumber"]
    .nunique()
    .sort_values(ascending=False)
)

tnved_4_support.head(30)

tnved_4
3304    28659
3305    12318
3401     8482
2710     8451
3403     5703
3303     3848
3307     3318
3402     1988
3820     1162
3808     1056
3306      991
3819      515
3301      112
8212       72
1905       67
3405       44
2106       31
1602       22
8481       19
9603       19
9031       19
8501       19
8421       18
0406       18
6104       18
8536       18
8708       18
8544       17
8482       17
8479       17
Name: rd_documentnumber, dtype: int64

In [85]:
tnved_4_candidates = (
    tnved_4_tg_long
    .groupby(["tnved_4", "tg"])["rd_documentnumber"]
    .nunique()
    .reset_index(name="docs")
)

tnved_4_candidates = tnved_4_candidates.merge(
    tnved_4_support.rename("total_docs"),
    on="tnved_4",
    how="left"
)

tnved_4_candidates["share"] = (
    tnved_4_candidates["docs"]
    / tnved_4_candidates["total_docs"]
)

tnved_4_candidates.sort_values(
    ["share", "total_docs"],
    ascending=[False, False]
).head(50)

,tnved_4,tg,docs,total_docs,share
291,8212,35,72,72,1.0
334,8481,43,19,19,1.0
376,8708,43,18,18,1.0
176,4016,43,17,17,1.0
336,8482,43,17,17,1.0
384,9026,43,17,17,1.0
392,9032,43,17,17,1.0
170,4009,43,16,16,1.0
277,7320,43,16,16,1.0
305,8409,43,16,16,1.0


In [86]:
tnved_feature_candidates = tnved_4_candidates.copy()

tnved_feature_candidates["purity"] = (
    tnved_feature_candidates["docs"]
    / tnved_feature_candidates["total_docs"]
)

In [87]:
tnved_feature_candidates["coverage"] = (
    tnved_feature_candidates["docs"]
    / tnved_feature_candidates["tg"].map(tg_docs)
)

In [88]:
tnved_feature_candidates.sort_values(
    ["tg", "purity", "coverage"],
    ascending=[True, False, False]
)

,tnved_4,tg,docs,total_docs,share,purity,coverage
291,8212,35,72,72,1.000000,1.000000,0.001053
60,1704,35,10,10,1.000000,1.000000,0.000146
75,2008,35,7,7,1.000000,1.000000,0.000102
85,2203,35,6,6,1.000000,1.000000,0.000088
99,3204,35,6,6,1.000000,1.000000,0.000088
...,...,...,...,...,...,...,...
123,3307,43,1,3318,0.000301,0.000301,0.000071
112,3303,43,1,3848,0.000260,0.000260,0.000071
115,3304,43,7,28659,0.000244,0.000244,0.000498
126,3401,43,1,8482,0.000118,0.000118,0.000071


## 3.x TNVED definitions validation

In [89]:
tnved_def = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ТНВЭД по категориям"
)

In [90]:
tnved_def.head(20)

,Парфимерия,Unnamed: 1
0,Код ТНВЭД,Наименование кода ТНВЭД
1,3303001000,3303001000 Духи
2,3303009000,3303009000 Туалетная вода
3,Косметика,NaN
4,3304100000,3304100000 Средства для макияжа губ
5,3304200000,3304200000 Средства для макияжа глаз
6,3304300000,3304300000 Средства для маникюра или педикюра
7,3304910000,"3304910000 Пудра, включая компактную"
8,3304990000,3304990000 Прочие косметические средства или с...
9,3305100000,3305100000 Шампуни


In [91]:
tnved_def.info()

<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Парфимерия  76 non-null     object
 1   Unnamed: 1  74 non-null     str   
dtypes: object(1), str(1)
memory usage: 12.5+ KB


In [92]:
code_col = tnved_def.columns[0]

not_codes = tnved_def.loc[
    ~tnved_def[code_col].astype("string").str.fullmatch(r"\d+"),
    [tnved_def.columns[0], tnved_def.columns[1]]
]

not_codes

,Парфимерия,Unnamed: 1
0,Код ТНВЭД,Наименование кода ТНВЭД
3,Косметика,NaN
68,Моторные масла,NaN


In [93]:
df = tnved_def.copy()

code_col = df.columns[0]
name_col = df.columns[1]

df["code_raw"] = df[code_col].astype("string").str.strip()

is_code = df["code_raw"].str.fullmatch(r"\d+")

# Служебный заголовок не является категорией
category_header = (
    ~is_code
    & df["code_raw"].notna()
    & (df["code_raw"] != "Код ТНВЭД")
)

df["category"] = df["code_raw"].where(category_header).ffill()

definitions_clean = (
    df.loc[is_code, ["category", "code_raw", name_col]]
    .rename(columns={
        "code_raw": "tnved_code",
        name_col: "tnved_name"
    })
    .reset_index(drop=True)
)

definitions_clean.head(20)

,category,tnved_code,tnved_name
0,<NA>,3303001000,3303001000 Духи
1,<NA>,3303009000,3303009000 Туалетная вода
2,Косметика,3304100000,3304100000 Средства для макияжа губ
3,Косметика,3304200000,3304200000 Средства для макияжа глаз
4,Косметика,3304300000,3304300000 Средства для маникюра или педикюра
5,Косметика,3304910000,"3304910000 Пудра, включая компактную"
6,Косметика,3304990000,3304990000 Прочие косметические средства или с...
7,Косметика,3305100000,3305100000 Шампуни
8,Косметика,3305200000,3305200000 Средства для перманентной завивки и...
9,Косметика,3305300000,3305300000 Лаки для волос


In [94]:
candidate_prefixes = [
    "3303",
    "3304",
    "3305",
    "3401",
    "2710",
    "3403",
]

In [95]:
definitions_clean["tnved_4"] = (
    definitions_clean["tnved_code"]
    .str[:4]
)

definitions_clean[
    definitions_clean["tnved_4"].isin(candidate_prefixes)
].sort_values(
    ["category", "tnved_4", "tnved_code"]
)

,category,tnved_code,tnved_name,tnved_4
2,Косметика,3304100000,3304100000 Средства для макияжа губ,3304
3,Косметика,3304200000,3304200000 Средства для макияжа глаз,3304
4,Косметика,3304300000,3304300000 Средства для маникюра или педикюра,3304
5,Косметика,3304910000,"3304910000 Пудра, включая компактную",3304
6,Косметика,3304990000,3304990000 Прочие косметические средства или с...,3304
7,Косметика,3305100000,3305100000 Шампуни,3305
8,Косметика,3305200000,3305200000 Средства для перманентной завивки и...,3305
9,Косметика,3305300000,3305300000 Лаки для волос,3305
10,Косметика,3305900001,3305900001 Лосьоны для волос,3305
11,Косметика,3305900009,3305900009 Прочие средства для волос,3305


In [96]:
(
    definitions_clean[
        definitions_clean["tnved_4"].isin(candidate_prefixes)
    ]
    .groupby(["tnved_4", "category"])["tnved_code"]
    .nunique()
)

tnved_4  category      
2710     Моторные масла    2
3304     Косметика         5
3305     Косметика         5
3401     Косметика         6
3403     Моторные масла    3
Name: tnved_code, dtype: int64

In [97]:
df = tnved_def.copy()

code_col = df.columns[0]
name_col = df.columns[1]

df["code_raw"] = (
    df[code_col]
    .astype("string")
    .str.strip()
)

is_code = df["code_raw"].str.fullmatch(r"\d+")

category_header = (
    ~is_code
    & df["code_raw"].notna()
    & (df["code_raw"] != "Код ТНВЭД")
)

# Начальная категория берётся из названия первого столбца Excel
df["category"] = df["code_raw"].where(category_header)

df.loc[df.index[0], "category"] = code_col

df["category"] = df["category"].ffill()

definitions_clean = (
    df.loc[
        is_code,
        ["category", "code_raw", name_col]
    ]
    .rename(columns={
        "code_raw": "tnved_code",
        name_col: "tnved_name"
    })
    .reset_index(drop=True)
)

definitions_clean["tnved_4"] = (
    definitions_clean["tnved_code"].str[:4]
)

In [98]:
definitions_clean[
    definitions_clean["tnved_4"].isin(candidate_prefixes)
].sort_values(
    ["category", "tnved_4", "tnved_code"]
)

,category,tnved_code,tnved_name,tnved_4
2,Косметика,3304100000,3304100000 Средства для макияжа губ,3304
3,Косметика,3304200000,3304200000 Средства для макияжа глаз,3304
4,Косметика,3304300000,3304300000 Средства для маникюра или педикюра,3304
5,Косметика,3304910000,"3304910000 Пудра, включая компактную",3304
6,Косметика,3304990000,3304990000 Прочие косметические средства или с...,3304
7,Косметика,3305100000,3305100000 Шампуни,3305
8,Косметика,3305200000,3305200000 Средства для перманентной завивки и...,3305
9,Косметика,3305300000,3305300000 Лаки для волос,3305
10,Косметика,3305900001,3305900001 Лосьоны для волос,3305
11,Косметика,3305900009,3305900009 Прочие средства для волос,3305


In [99]:
definitions_clean[
    definitions_clean["tnved_4"].isin(candidate_prefixes)
].groupby(
    ["tnved_4", "category"]
)["tnved_code"].nunique()

tnved_4  category      
2710     Моторные масла    2
3303     Парфимерия        2
3304     Косметика         5
3305     Косметика         5
3401     Косметика         6
3403     Моторные масла    3
Name: tnved_code, dtype: int64

In [100]:
print("Заполнено:")
print(base_df["product_origin"].notna().mean())

print("\nТоп стран:")
base_df["product_origin"].value_counts(dropna=False).head(30)

Заполнено:
0.7503389300239858

Топ стран:


product_origin
РОССИЯ                           34844
NaN                              21546
КОРЕЯ, РЕСПУБЛИКА                 5879
ФРАНЦИЯ                           5362
ИТАЛИЯ                            3456
ГЕРМАНИЯ                          2985
ЯПОНИЯ                            1787
КИТАЙ                             1566
ИСПАНИЯ                           1426
ТУРЦИЯ                            1363
СОЕДИНЕННЫЕ ШТАТЫ                 1105
СОЕДИНЕННОЕ КОРОЛЕВСТВО            715
ПОЛЬША                             658
ИЗРАИЛЬ                            411
НИДЕРЛАНДЫ                         383
ТАИЛАНД                            287
ГРЕЦИЯ                             284
ИНДИЯ                              246
ИРЛАНДИЯ                           177
ЛЮКСЕМБУРГ                         170
БЕЛЬГИЯ                            162
ШВЕЦИЯ                             135
ЧЕХИЯ                              119
ФИНЛЯНДИЯ                           96
СИНГАПУР                            93
ОБЪЕДИНЕНН

In [101]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
).copy()

In [102]:
origin_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["product_origin"].fillna("Нет данных"),
    normalize="index"
)

origin_by_tg

product_origin,АБХАЗИЯ,АВСТРАЛИЯ,АВСТРИЯ,АЗЕРБАЙДЖАН,БЕЛАРУСЬ,БЕЛЬГИЯ,БОЛГАРИЯ,БОСНИЯ И ГЕРЦЕГОВИНА,БРАЗИЛИЯ,ВАНУАТУ,...,ФИЛИППИНЫ,ФИНЛЯНДИЯ,ФРАНЦИЯ,ЧЕХИЯ,ШВЕЙЦАРИЯ,ШВЕЦИЯ,ШРИ-ЛАНКА,ЭСТОНИЯ,ЮЖНАЯ АФРИКА,ЯПОНИЯ
tg,,,,,,,,,,,,,,,,,,,,,
35,0.000000,0.000044,0.000015,0.000029,0.000410,0.000190,0.000015,0.000000,0.000073,0.000015,...,0.000015,0.000951,0.058944,0.001565,0.000380,0.000878,0.000614,0.000278,0.000059,0.017054
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.239329,0.002795,0.002287,0.000254,0.000000,0.000000,0.000000,0.000254
43,0.000142,0.000000,0.000925,0.000427,0.000356,0.010597,0.000071,0.000071,0.000000,0.000000,...,0.000000,0.002205,0.027807,0.000071,0.002205,0.005263,0.000000,0.000000,0.000071,0.044094


In [103]:
tg_exploded["tg_str"] = (
    tg_exploded["tg"]
    .astype("string")
)

In [104]:
origin_top = pd.crosstab(
    tg_exploded["tg_str"],
    tg_exploded["product_origin"].fillna("Нет данных"),
    normalize="index"
)

for tg in ["4", "35", "43"]:
    print(f"\n=== TG {tg} ===")
    print(
        origin_top.loc[tg]
        .sort_values(ascending=False)
        .head(15)
    )


=== TG 4 ===
product_origin
Нет данных                       0.275661
РОССИЯ                           0.266514
ФРАНЦИЯ                          0.239329
ИСПАНИЯ                          0.072663
ИТАЛИЯ                           0.051321
ТУРЦИЯ                           0.020071
ГЕРМАНИЯ                         0.013465
СОЕДИНЕННОЕ КОРОЛЕВСТВО          0.012195
СОЕДИНЕННЫЕ ШТАТЫ                0.010163
ПОЛЬША                           0.007368
КОРЕЯ, РЕСПУБЛИКА                0.004065
ОБЪЕДИНЕННЫЕ АРАБСКИЕ ЭМИРАТЫ    0.003811
НИДЕРЛАНДЫ                       0.003303
КИТАЙ                            0.003303
ОМАН                             0.003049
Name: 4, dtype: float64

=== TG 35 ===
product_origin
РОССИЯ                     0.401667
Нет данных                 0.260290
КОРЕЯ, РЕСПУБЛИКА          0.079552
ФРАНЦИЯ                    0.058944
ИТАЛИЯ                     0.045634
ГЕРМАНИЯ                   0.026371
КИТАЙ                      0.017669
ЯПОНИЯ                     0.017054

In [105]:
origin_precision = pd.crosstab(
    tg_exploded["product_origin"].fillna("Нет данных"),
    tg_exploded["tg_str"],
    normalize="index"
)

In [106]:
origin_support = (
    tg_exploded
    .groupby("product_origin")["rd_documentnumber"]
    .nunique()
    .sort_values(ascending=False)
)

origin_support.head(30)

product_origin
РОССИЯ                           34844
КОРЕЯ, РЕСПУБЛИКА                 5879
ФРАНЦИЯ                           5362
ИТАЛИЯ                            3456
ГЕРМАНИЯ                          2985
ЯПОНИЯ                            1787
КИТАЙ                             1566
ИСПАНИЯ                           1426
ТУРЦИЯ                            1363
СОЕДИНЕННЫЕ ШТАТЫ                 1105
СОЕДИНЕННОЕ КОРОЛЕВСТВО            715
ПОЛЬША                             658
ИЗРАИЛЬ                            411
НИДЕРЛАНДЫ                         383
ТАИЛАНД                            287
ГРЕЦИЯ                             284
ИНДИЯ                              246
ИРЛАНДИЯ                           177
ЛЮКСЕМБУРГ                         170
БЕЛЬГИЯ                            162
ШВЕЦИЯ                             135
ЧЕХИЯ                              119
ФИНЛЯНДИЯ                           96
СИНГАПУР                            93
ОБЪЕДИНЕННЫЕ АРАБСКИЕ ЭМИРАТЫ       75
ИРАН, ИСЛА

In [107]:
origin_precision_supported = origin_precision.loc[
    origin_precision.index.isin(
        origin_support[origin_support >= 100].index
    )
]

origin_precision_supported.sort_values(
    "4",
    ascending=False
).head(20)

tg_str,35,4,43
product_origin,,,
ИСПАНИЯ,0.762973,0.200561,0.036466
ФРАНЦИЯ,0.751445,0.175648,0.072907
ЧЕХИЯ,0.899160,0.092437,0.008403
СОЕДИНЕННОЕ КОРОЛЕВСТВО,0.539106,0.067039,0.393855
ИТАЛИЯ,0.902517,0.058432,0.039051
ТУРЦИЯ,0.688645,0.057875,0.253480
ИРЛАНДИЯ,0.937853,0.045198,0.016949
ПОЛЬША,0.931715,0.044006,0.024279
СОЕДИНЕННЫЕ ШТАТЫ,0.640541,0.036036,0.323423


In [108]:
origin_precision_supported.sort_values(
    "35",
    ascending=False
).head(20)

tg_str,35,4,43
product_origin,,,
ИЗРАИЛЬ,0.997567,0.002433,0.000000
ЛЮКСЕМБУРГ,0.988235,0.011765,0.000000
ИРЛАНДИЯ,0.937853,0.045198,0.016949
ПОЛЬША,0.931715,0.044006,0.024279
"КОРЕЯ, РЕСПУБЛИКА",0.925157,0.002722,0.072121
ИТАЛИЯ,0.902517,0.058432,0.039051
ЧЕХИЯ,0.899160,0.092437,0.008403
ТАИЛАНД,0.874564,0.000000,0.125436
ИНДИЯ,0.869919,0.004065,0.126016


In [109]:
origin_precision_supported.sort_values(
    "43",
    ascending=False
).head(20)

tg_str,35,4,43
product_origin,,,
БЕЛЬГИЯ,0.080247,0.000000,0.919753
НИДЕРЛАНДЫ,0.381201,0.033943,0.584856
ШВЕЦИЯ,0.444444,0.007407,0.548148
СОЕДИНЕННОЕ КОРОЛЕВСТВО,0.539106,0.067039,0.393855
ГЕРМАНИЯ,0.603212,0.017732,0.379057
ЯПОНИЯ,0.652490,0.000560,0.346950
СОЕДИНЕННЫЕ ШТАТЫ,0.640541,0.036036,0.323423
ТУРЦИЯ,0.688645,0.057875,0.253480
КИТАЙ,0.771392,0.008301,0.220307


In [110]:
origin_manufacturer = pd.crosstab(
    tg_exploded["product_origin"].fillna("Нет данных"),
    tg_exploded["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

In [111]:
origin_manufacturer = pd.crosstab(
    tg_exploded["product_origin"].fillna("Нет данных"),
    tg_exploded["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

origin_manufacturer.head(20)

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
product_origin,,,,,
АБХАЗИЯ,0.000000,1.000000,0.0,0.000000,0.000000
АВСТРАЛИЯ,0.000000,0.333333,0.0,0.666667,0.000000
АВСТРИЯ,0.000000,0.714286,0.0,0.285714,0.000000
АЗЕРБАЙДЖАН,0.000000,0.375000,0.0,0.625000,0.000000
БЕЛАРУСЬ,0.030303,0.757576,0.0,0.121212,0.090909
БЕЛЬГИЯ,0.000000,0.302469,0.0,0.697531,0.000000
БОЛГАРИЯ,0.000000,0.500000,0.0,0.500000,0.000000
БОСНИЯ И ГЕРЦЕГОВИНА,0.000000,0.000000,0.0,1.000000,0.000000
БРАЗИЛИЯ,0.000000,0.200000,0.0,0.800000,0.000000


In [112]:
countries_to_check = [
    "РОССИЯ",
    "ФРАНЦИЯ",
    "ГЕРМАНИЯ",
    "ИТАЛИЯ",
    "ЯПОНИЯ",
    "БЕЛЬГИЯ"
]

origin_manufacturer.loc[
    origin_manufacturer.index.isin(countries_to_check)
]

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
product_origin,,,,,
БЕЛЬГИЯ,0.000000,0.302469,0.0,0.697531,0.000000
ГЕРМАНИЯ,0.000000,0.935095,0.0,0.064905,0.000000
ИТАЛИЯ,0.000000,0.966734,0.0,0.031530,0.001736
РОССИЯ,0.110082,0.000516,0.0,0.000000,0.889402
ФРАНЦИЯ,0.000000,0.930636,0.0,0.068059,0.001305
ЯПОНИЯ,0.000000,0.942921,0.0,0.057079,0.000000


In [113]:
text_fields = [
    "nameProd",
    "docNorm",
    "useArea",
    "RD_fullname",
    "protocol",
    "firmGetName",
    "firmMadeName",
    "statusGroup",
    "techRegulations",
]

text_stats = {}

for field in text_fields:
    values = []

    for row in base_df["rd_data"]:
        data = json.loads(row)
        value = data.get(field)

        values.append(
            isinstance(value, str) and bool(value.strip())
        )

    text_stats[field] = {
        "present": sum(values),
        "coverage": sum(values) / len(values)
    }

text_stats_df = (
    pd.DataFrame(text_stats)
    .T
    .sort_values("coverage", ascending=False)
)

text_stats_df

,present,coverage
techRegulations,70358.0,0.815263
nameProd,11362.0,0.131655
protocol,11361.0,0.131644
firmGetName,11361.0,0.131644
firmMadeName,11355.0,0.131574
useArea,11233.0,0.130161
docNorm,10648.0,0.123382
RD_fullname,0.0,0.000000
statusGroup,0.0,0.000000


In [114]:
tech_reg_values = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    value = data.get("techRegulations")

    if isinstance(value, str) and value.strip():
        tech_reg_values.append(value.strip())
    else:
        tech_reg_values.append(None)

base_df["tech_regulations"] = tech_reg_values

In [115]:
base_df["tech_regulations"].value_counts(dropna=False).head(30)

tech_regulations
ТР ТС 009/2011 О безопасности парфюмерно-косметической продукции                                                                                                                                                                                                                             56174
NaN                                                                                                                                                                                                                                                                                          15943
ТР ТС 030/2012 О требованиях к смазочным материалам, маслам и специальным жидкостям                                                                                                                                                                                                          13508
ТР ТС 024/2011 Технический регламент на масложировую продукцию                                                

In [116]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
).copy()

tg_exploded["tg_str"] = tg_exploded["tg"].astype("string")

In [117]:
tech_reg_by_tg = pd.crosstab(
    tg_exploded["tg_str"],
    tg_exploded["tech_regulations"].fillna("Нет данных"),
    normalize="index"
)

for tg in ["4", "35", "43"]:
    print(f"\n=== TG {tg} ===")
    print(
        tech_reg_by_tg.loc[tg]
        .sort_values(ascending=False)
        .head(20)
    )


=== TG 4 ===
tech_regulations
ТР ТС 009/2011 О безопасности парфюмерно-косметической продукции                                                                                                                                                                                                                        0.981199
Нет данных                                                                                                                                                                                                                                                                              0.012449
ТР ТС 017/2011 О безопасности продукции легкой промышленности                                                                                                                                                                                                                           0.001270
ТР ТС 007/2011 О безопасности продукции, предназначенной для детей и подростков                       

In [118]:
base_df.loc[
    base_df["tech_regulations"].notna(),
    ["rd_documentnumber", "tech_regulations"]
].sample(20, random_state=42)

,rd_documentnumber,tech_regulations
77307,ЕАЭС N RU Д-RU.РА10.В.60655/24,ТР ТС 030/2012 О требованиях к смазочным матер...
30347,ЕАЭС N RU Д-FR.РА10.В.00518/24,ТР ТС 009/2011 О безопасности парфюмерно-косме...
43431,ЕАЭС N RU Д-KR.РА09.В.77965/23,ТР ТС 009/2011 О безопасности парфюмерно-косме...
50226,ЕАЭС N RU Д-RU.РА01.В.81165/23,ТР ТС 009/2011 О безопасности парфюмерно-косме...
51923,ЕАЭС N RU Д-RU.РА02.В.27363/25,ТР ТС 009/2011 О безопасности парфюмерно-косме...
35582,ЕАЭС N RU Д-IT.РА08.В.23513/22,ТР ТС 009/2011 О безопасности парфюмерно-косме...
69990,ЕАЭС N RU Д-RU.РА07.В.58174/23,ТР ТС 009/2011 О безопасности парфюмерно-косме...
71187,ЕАЭС N RU Д-RU.РА08.В.01986/25,ТР ТС 030/2012 О требованиях к смазочным матер...
60667,ЕАЭС N RU Д-RU.РА04.В.64611/25,ТР ТС 009/2011 О безопасности парфюмерно-косме...
29644,ЕАЭС N RU Д-FR.РА07.В.69359/24,ТР ТС 009/2011 О безопасности парфюмерно-косме...


In [119]:
import re

def extract_tr_codes(value):
    if not isinstance(value, str):
        return []

    return sorted(
        set(
            re.findall(
                r"ТР\s+(?:ТС|ЕАЭС)\s+\d{3}/\d{4}",
                value
            )
        )
    )

base_df["tech_reg_codes"] = (
    base_df["tech_regulations"]
    .apply(extract_tr_codes)
)

In [120]:
base_df["tech_reg_codes"].apply(len).value_counts().sort_index()

tech_reg_codes
0    15943
1    70066
2       50
3      185
4       56
5        1
Name: count, dtype: int64

In [121]:
base_df["tech_reg_codes"].explode().value_counts().head(30)

tech_reg_codes
ТР ТС 009/2011      56175
ТР ТС 030/2012      13508
ТР ТС 021/2011        289
ТР ТС 022/2011        286
ТР ТС 029/2012        199
ТР ТС 024/2011        153
ТР ТС 019/2011        104
ТР ТС 017/2011         61
ТР ТС 033/2013         43
ТР ТС 034/2013         31
ТР ТС 007/2011         19
ТР ТС 010/2011         14
ТР ТС 018/2011         12
ТР ЕАЭС 040/2016       11
ТР ТС 005/2011          8
ТР ТС 020/2011          6
ТР ТС 004/2011          5
ТР ТС 011/2011          4
ТР ЕАЭС 037/2016        4
ТР ЕАЭС 044/2017        4
ТР ТС 023/2011          4
ТР ТС 032/2013          2
ТР ТС 014/2011          2
ТР ТС 013/2011          2
ТР ТС 008/2011          1
ТР ТС 015/2011          1
ТР ТС 035/2014          1
ТР ТС 025/2012          1
Name: count, dtype: int64

In [122]:
tech_reg_base = base_df[
    ["rd_documentnumber", "tech_reg_codes"]
].rename(
    columns={"rd_documentnumber": "rd_number"}
)

tech_reg_tg = doc_tg_type.merge(
    tech_reg_base,
    on="rd_number",
    how="inner"
)

In [123]:
tech_reg_tg_long = (
    tech_reg_tg
    .explode("tech_reg_codes", ignore_index=True)
    .dropna(subset=["tech_reg_codes"])
    .drop_duplicates(
        subset=["rd_number", "tg", "tech_reg_codes"]
    )
)

In [124]:
tech_reg_precision = pd.crosstab(
    tech_reg_tg_long["tech_reg_codes"],
    tech_reg_tg_long["tg"],
    normalize="index"
)

tech_reg_precision

tg,35,4,43
tech_reg_codes,,,
ТР ЕАЭС 037/2016,1.000000,0.000000,0.000000
ТР ЕАЭС 040/2016,0.909091,0.000000,0.090909
ТР ЕАЭС 044/2017,1.000000,0.000000,0.000000
ТР ТС 004/2011,0.600000,0.000000,0.400000
ТР ТС 005/2011,0.500000,0.000000,0.500000
ТР ТС 007/2011,0.736842,0.210526,0.052632
ТР ТС 008/2011,1.000000,0.000000,0.000000
ТР ТС 009/2011,0.931150,0.068708,0.000142
ТР ТС 010/2011,1.000000,0.000000,0.000000


In [125]:
tech_reg_support = (
    tech_reg_tg_long
    .groupby("tech_reg_codes")["rd_number"]
    .nunique()
    .sort_values(ascending=False)
)

tech_reg_support.head(20)

tech_reg_codes
ТР ТС 009/2011      56175
ТР ТС 030/2012      13508
ТР ТС 021/2011        289
ТР ТС 022/2011        286
ТР ТС 029/2012        199
ТР ТС 024/2011        153
ТР ТС 019/2011        104
ТР ТС 017/2011         61
ТР ТС 033/2013         43
ТР ТС 034/2013         31
ТР ТС 007/2011         19
ТР ТС 010/2011         14
ТР ТС 018/2011         12
ТР ЕАЭС 040/2016       11
ТР ТС 005/2011          8
ТР ТС 020/2011          6
ТР ТС 004/2011          5
ТР ЕАЭС 044/2017        4
ТР ЕАЭС 037/2016        4
ТР ТС 011/2011          4
Name: rd_number, dtype: int64

In [126]:
candidate_interactions = (
    tech_reg_tg_long[
        tech_reg_tg_long["tech_reg_codes"].isin([
            "ТР ТС 009/2011",
            "ТР ТС 030/2012"
        ])
    ][
        ["rd_number", "tg", "tech_reg_codes"]
    ]
)

In [127]:
interaction_df = (
    tnved_4_tg_long[
        ["rd_documentnumber", "tg", "tnved_4"]
    ]
    .rename(columns={"rd_documentnumber": "rd_number"})
    .merge(
        tech_reg_tg_long[
            ["rd_number", "tg", "tech_reg_codes"]
        ],
        on=["rd_number", "tg"],
        how="inner"
    )
    .drop_duplicates()
)

In [128]:
interaction_df[
    interaction_df["tech_reg_codes"].isin([
        "ТР ТС 009/2011",
        "ТР ТС 030/2012"
    ])
].groupby(
    ["tech_reg_codes", "tnved_4", "tg"]
)["rd_number"].nunique().reset_index(name="docs")

,tech_reg_codes,tnved_4,tg,docs
0,ТР ТС 009/2011,1302,35,1
1,ТР ТС 009/2011,1515,35,1
2,ТР ТС 009/2011,1521,35,2
3,ТР ТС 009/2011,2106,35,5
4,ТР ТС 009/2011,2933,35,1
5,ТР ТС 009/2011,3301,35,42
6,ТР ТС 009/2011,3301,4,69
7,ТР ТС 009/2011,3302,4,10
8,ТР ТС 009/2011,3303,35,43
9,ТР ТС 009/2011,3303,4,3770


In [129]:
anchor_tnved = {
    "tg4_3303": "3303",
    "tg35_3304": "3304",
    "tg35_3305": "3305",
    "tg35_3401": "3401",
    "tg43_2710": "2710",
    "tg43_3403": "3403",
}

for feature_name, code in anchor_tnved.items():
    base_df[feature_name] = base_df["tnved_4"].apply(
        lambda codes: code in codes
    )

In [130]:
base_df["tr_009"] = base_df["tech_reg_codes"].apply(
    lambda codes: "ТР ТС 009/2011" in codes
)

base_df["tr_030"] = base_df["tech_reg_codes"].apply(
    lambda codes: "ТР ТС 030/2012" in codes
)

In [131]:
tg_anchor = base_df.explode(
    "tg",
    ignore_index=True
).copy()

tg_anchor["tg"] = tg_anchor["tg"].astype("string")

In [132]:
anchor_columns = [
    "tg4_3303",
    "tg35_3304",
    "tg35_3305",
    "tg35_3401",
    "tg43_2710",
    "tg43_3403",
    "tr_009",
    "tr_030",
]

anchor_coverage = (
    tg_anchor
    .groupby("tg")[anchor_columns]
    .mean()
)

anchor_coverage

,tg4_3303,tg35_3304,tg35_3305,tg35_3401,tg43_2710,tg43_3403,tr_009,tr_030
tg,,,,,,,,
35,0.000644,0.418385,0.180152,0.123870,0.000102,0.000059,0.765526,0.000176
4,0.970020,0.012703,0.000762,0.005335,0.000000,0.000254,0.981199,0.000254
43,0.000071,0.000498,0.000071,0.000071,0.600740,0.405448,0.000569,0.960174


In [133]:
tg_anchor["anchor_tg4"] = (
    tg_anchor["tg4_3303"]
    | (
        tg_anchor["tr_009"]
        & tg_anchor["tg4_3303"]
    )
)

In [134]:
tg_anchor["anchor_tg4"] = tg_anchor["tg4_3303"]

In [135]:
tg_anchor["anchor_tg35"] = (
    tg_anchor["tg35_3304"]
    | tg_anchor["tg35_3305"]
    | tg_anchor["tg35_3401"]
)

In [136]:
tg_anchor["anchor_tg43"] = (
    tg_anchor["tg43_2710"]
    | tg_anchor["tg43_3403"]
)

In [137]:
pd.DataFrame({
    "TG4": [
        tg_anchor.loc[tg_anchor["tg"] == "4", "anchor_tg4"].mean()
    ],
    "TG35": [
        tg_anchor.loc[tg_anchor["tg"] == "35", "anchor_tg35"].mean()
    ],
    "TG43": [
        tg_anchor.loc[tg_anchor["tg"] == "43", "anchor_tg43"].mean()
    ],
})

,TG4,TG35,TG43
0,0.97002,0.720067,0.872129


In [138]:
tg_anchor["tg43_structural"] = (
    tg_anchor["tg43_2710"]
    | tg_anchor["tg43_3403"]
    | tg_anchor["tr_030"]
)

In [139]:
tg_anchor.loc[
    tg_anchor["tg"] == "43",
    "tg43_structural"
].mean()

np.float64(0.9814380200554725)

In [140]:
tg_anchor["n_anchor_groups"] = (
    tg_anchor[
        ["tg4_3303", "tg35_3304", "tg35_3305", "tg35_3401",
         "tg43_2710", "tg43_3403"]
    ]
    .astype(int)
    .sum(axis=1)
)

tg_anchor["n_anchor_groups"].value_counts().sort_index()

n_anchor_groups
0    20939
1    63383
2     2022
3       22
4        1
Name: count, dtype: int64

In [141]:
tg_anchor.loc[
    tg_anchor["n_anchor_groups"] >= 2,
    [
        "rd_documentnumber",
        "tg",
        "tg4_3303",
        "tg35_3304",
        "tg35_3305",
        "tg35_3401",
        "tg43_2710",
        "tg43_3403"
    ]
].head(30)

,rd_documentnumber,tg,tg4_3303,tg35_3304,tg35_3305,tg35_3401,tg43_2710,tg43_3403
11411,ВП RU Д-BE.РА01.А.40149/25,43,False,False,False,False,True,True
11417,ВП RU Д-BE.РА01.А.80023/25,43,False,False,False,False,True,True
11446,ВП RU Д-CN.РА01.А.02069/23,4,True,True,False,True,False,False
11447,ВП RU Д-CN.РА01.А.03729/23,4,True,True,False,True,False,False
11501,ВП RU Д-CN.РА01.А.68108/25,35,False,True,True,True,False,False
11581,ВП RU Д-DE.РА01.А.26334/25,43,False,False,False,False,True,True
11587,ВП RU Д-DE.РА01.А.36326/25,43,False,False,False,False,True,True
11601,ВП RU Д-DE.РА01.А.66597/25,43,False,False,False,False,True,True
11618,ВП RU Д-DE.РА01.А.79219/23,43,False,False,False,False,True,True
11622,ВП RU Д-DE.РА01.А.90445/25,43,False,False,False,False,True,True


In [142]:
tg35_uncovered = tg_anchor[
    (tg_anchor["tg"] == "35") &
    (~tg_anchor["anchor_tg35"])
]

In [143]:
tg_anchor["anchor_tg4"] = (
    tg_anchor["tg4_3303"]
)

tg_anchor["anchor_tg35"] = (
    tg_anchor["tg35_3304"]
    | tg_anchor["tg35_3305"]
    | tg_anchor["tg35_3401"]
)

tg_anchor["anchor_tg43"] = (
    tg_anchor["tg43_2710"]
    | tg_anchor["tg43_3403"]
)

tg_anchor["n_anchor_target_groups"] = (
    tg_anchor[
        ["anchor_tg4", "anchor_tg35", "anchor_tg43"]
    ]
    .astype(int)
    .sum(axis=1)
)

tg_anchor["n_anchor_target_groups"].value_counts().sort_index()

n_anchor_target_groups
0    20939
1    65408
2       20
Name: count, dtype: int64

In [144]:
anchor_conflicts = tg_anchor[
    tg_anchor["n_anchor_target_groups"] >= 2
][
    [
        "rd_documentnumber",
        "tg",
        "anchor_tg4",
        "anchor_tg35",
        "anchor_tg43",
    ]
]

anchor_conflicts.head(30)

,rd_documentnumber,tg,anchor_tg4,anchor_tg35,anchor_tg43
11446,ВП RU Д-CN.РА01.А.02069/23,4,True,True,False
11447,ВП RU Д-CN.РА01.А.03729/23,4,True,True,False
16600,ЕАЭС N RU Д-CH.РА02.В.36933/24,4,True,True,False
16697,ЕАЭС N RU Д-CH.РА04.В.12836/23,4,True,True,False
17638,ЕАЭС N RU Д-CN.РА02.В.98436/25,4,True,True,False
17987,ЕАЭС N RU Д-CN.РА04.В.10891/23,4,True,True,False
19454,ЕАЭС N RU Д-CN.РА08.В.97241/23,4,True,True,False
30029,ЕАЭС N RU Д-FR.РА08.В.87908/24,4,True,True,False
46734,ЕАЭС N RU Д-RU.РА01.В.03832/24,4,True,True,False
51641,ЕАЭС N RU Д-RU.РА02.В.15977/25,35,True,True,False


In [145]:
anchor_conflicts["tg"].value_counts()

tg
4     15
35     4
43     1
Name: count, dtype: int64[pyarrow]

In [146]:
single_tg_conflicts = anchor_conflicts[
    anchor_conflicts["tg"].apply(
        lambda x: len(x) == 1
    )
]

len(single_tg_conflicts)

15

In [147]:
doc_anchor_check = (
    tg_anchor
    .groupby("rd_documentnumber")
    .agg(
        truth_tgs=("tg", lambda x: tuple(sorted(set(x.astype(str))))),
        anchor_tg4=("anchor_tg4", "first"),
        anchor_tg35=("anchor_tg35", "first"),
        anchor_tg43=("anchor_tg43", "first"),
    )
    .reset_index()
)

doc_anchor_check["n_anchor_target_groups"] = (
    doc_anchor_check[
        ["anchor_tg4", "anchor_tg35", "anchor_tg43"]
    ]
    .astype(int)
    .sum(axis=1)
)

In [148]:
doc_anchor_conflicts = doc_anchor_check[
    doc_anchor_check["n_anchor_target_groups"] >= 2
]

print("Конфликтующих документов:", len(doc_anchor_conflicts))

doc_anchor_conflicts[
    [
        "rd_documentnumber",
        "truth_tgs",
        "anchor_tg4",
        "anchor_tg35",
        "anchor_tg43",
    ]
]

Конфликтующих документов: 20


,rd_documentnumber,truth_tgs,anchor_tg4,anchor_tg35,anchor_tg43
11429,ВП RU Д-CN.РА01.А.02069/23,['4'],True,True,False
11430,ВП RU Д-CN.РА01.А.03729/23,['4'],True,True,False
15701,ЕАЭС N RU Д-CH.РА02.В.36933/24,['4'],True,True,False
15798,ЕАЭС N RU Д-CH.РА04.В.12836/23,['4'],True,True,False
16739,ЕАЭС N RU Д-CN.РА02.В.98436/25,['4'],True,True,False
17088,ЕАЭС N RU Д-CN.РА04.В.10891/23,['4'],True,True,False
18554,ЕАЭС N RU Д-CN.РА08.В.97241/23,['4'],True,True,False
29125,ЕАЭС N RU Д-FR.РА08.В.87908/24,['4'],True,True,False
45820,ЕАЭС N RU Д-RU.РА01.В.03832/24,['4'],True,True,False
50719,ЕАЭС N RU Д-RU.РА02.В.15977/25,['35'],True,True,False


In [149]:
doc_anchor_conflicts["truth_tgs_tuple"] = (
    doc_anchor_conflicts["truth_tgs"]
    .map(lambda x: tuple(x))
)

doc_anchor_conflicts["truth_tgs_tuple"].value_counts()

truth_tgs_tuple
(4,)     15
(35,)     4
(43,)     1
Name: count, dtype: int64

In [150]:
tg35_uncovered = tg_anchor[
    (tg_anchor["tg"] == "35") &
    (~tg_anchor["anchor_tg35"])
]

In [151]:
conflict_numbers = (
    doc_anchor_conflicts["rd_documentnumber"]
    .tolist()
)

conflict_docs = base_df.loc[
    base_df["rd_documentnumber"].isin(conflict_numbers),
    [
        "rd_documentnumber",
        "tg",
        "product_tnved",
        "product_name",
        "product_info",
        "tech_regulations",
        "rd_type",
    ]
]

conflict_docs

,rd_documentnumber,tg,product_tnved,product_name,product_info,tech_regulations,rd_type
11429,ВП RU Д-CN.РА01.А.02069/23,[4],"3303009000, 3304100000, 3401209000",Продукция косметическая:,"Контракт № 30122020 от 30.12.2020, Инвойс...",NaN,[N/A]
11430,ВП RU Д-CN.РА01.А.03729/23,[4],"3303009000, 3304990000, 3401300000",Продукция косметическая:,"Контракт № 30122020 от 30.12.2020, Инвойс...",NaN,[N/A]
16580,ЕАЭС N RU Д-CH.РА02.В.36933/24,[4],"330300, 3304990000",Парфюмерно-косметическая продукция: Селективны...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
16677,ЕАЭС N RU Д-CH.РА04.В.12836/23,[4],"330300, 3304990000",Парфюмерно-косметическая продукция: Селективны...,,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
17618,ЕАЭС N RU Д-CN.РА02.В.98436/25,[4],"3303001000, 3304990000",Косметическая и парфюмерная продукция для прид...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
17967,ЕАЭС N RU Д-CN.РА04.В.10891/23,[4],"3303001000, 3304990000",Косметическая и парфюмерная продукция для прид...,,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
19433,ЕАЭС N RU Д-CN.РА08.В.97241/23,[4],"3303001000, 3304990000",Косметическая и парфюмерная продукция для прид...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
30004,ЕАЭС N RU Д-FR.РА08.В.87908/24,[4],"3303001000, 3304990000",Парфюмерно косметическая продукция: духи для м...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
46699,ЕАЭС N RU Д-RU.РА01.В.03832/24,[4],"3303001000, 3304990000",Косметическая и парфюмерная продукция для прид...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[N/A]
51598,ЕАЭС N RU Д-RU.РА02.В.15977/25,[35],"3303001000, 3304990000, 3305100000, 3305900009",Изделия косметические для взрослых: Крем для в...,Декларация соответствия распространяется ...,ТР ТС 009/2011 О безопасности парфюмерно-косме...,[ДС]


TNVED anchor families могут одновременно срабатывать на одном РД; это особенно характерно для границы TG4/TG35. Поэтому anchors являются evidence, а не взаимоисключающими label rules.

In [152]:
anchor_conflict_types = (
    doc_anchor_conflicts[
        ["anchor_tg4", "anchor_tg35", "anchor_tg43"]
    ]
    .astype(int)
)

anchor_conflict_types.value_counts()

anchor_tg4  anchor_tg35  anchor_tg43
1           1            0              19
0           1            1               1
Name: count, dtype: int64

In [153]:
tg35_uncovered = tg_anchor[
    (tg_anchor["tg"] == "35")
    & (~tg_anchor["anchor_tg35"])
].copy()

print(tg35_uncovered.shape)
print(tg35_uncovered["rd_documentnumber"].nunique())

(19139, 44)
19139


In [154]:
tg35_all = tg_anchor[
    tg_anchor["tg"] == "35"
].copy()

tg35_covered = tg35_all[
    tg35_all["anchor_tg35"]
].copy()

tg35_uncovered = tg35_all[
    ~tg35_all["anchor_tg35"]
].copy()

print("Всего TG35:", len(tg35_all))
print("Покрыто anchors:", len(tg35_covered))
print("Не покрыто:", len(tg35_uncovered))

print(
    "Coverage:",
    len(tg35_covered) / len(tg35_all)
)
print(
    "Uncovered share:",
    len(tg35_uncovered) / len(tg35_all)
)

Всего TG35: 68370
Покрыто anchors: 49231
Не покрыто: 19139
Coverage: 0.7200672809711862
Uncovered share: 0.2799327190288138


In [155]:
tg35_uncovered["tnved_count"] = (
    tg35_uncovered["tnved_list"]
    .apply(len)
)

tg35_uncovered["tnved_count"].value_counts().sort_index()

tnved_count
0      11322
1       7591
2        160
3         24
4         12
5         10
6          3
7          4
8          2
9          2
11         1
13         1
15         1
16         2
20         1
23         2
220        1
Name: count, dtype: int64

In [156]:
tg35_uncovered["tnved_count"].describe()

count    19139.000000
mean         0.445060
std          1.719379
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max        220.000000
Name: tnved_count, dtype: float64

In [157]:
tg35_uncovered["tnved_list"].explode().value_counts().head(30)

tnved_list
3402500000    1329
3307200000    1142
3307300000     850
3307100000     751
3808948000     685
3306100000     602
3306900000     381
3402909000     296
3402           288
3307900008     218
3307490000     207
3808941000     152
380894         142
3402209000      76
3307            63
3808943000      50
8212109000      33
8212101000      30
3303009000      28
3808            27
1905            25
8212200000      21
1905909000      18
340290          18
3405            16
3405400000      16
3301909000      15
3303001000      12
330790000       11
1905906000      11
Name: count, dtype: int64

In [158]:
tg35_uncovered_tnved = (
    tg35_uncovered[
        ["rd_documentnumber", "tnved_4"]
    ]
    .explode("tnved_4", ignore_index=True)
    .dropna(subset=["tnved_4"])
    .drop_duplicates(
        subset=["rd_documentnumber", "tnved_4"]
    )
)

In [159]:
tg35_uncovered_tnved["tnved_4"].value_counts().head(30)

tnved_4
3307    3230
3402    1973
3808    1055
3306     989
8212      72
1905      60
3303      40
3405      40
3301      36
2106      28
1602      19
6104      13
0406      13
6110      11
1704      10
1902      10
6103       9
6106       9
1103       8
0403       8
1601       8
3814       8
6204       7
6105       7
6112       7
2710       7
2008       7
1604       6
1806       6
2203       6
Name: count, dtype: int64

In [160]:
new_tnved_candidates = [
    "3307",
    "3402",
    "3808",
    "3306",
]

tg35_uncovered_tnved["tnved_4"].value_counts().loc[
    new_tnved_candidates
]

tnved_4
3307    3230
3402    1973
3808    1055
3306     989
Name: count, dtype: int64

In [161]:
candidate_eval = (
    tnved_4_tg_long[
        tnved_4_tg_long["tnved_4"].isin(new_tnved_candidates)
    ]
    .groupby(
        ["tnved_4", "tg"]
    )["rd_documentnumber"]
    .nunique()
    .reset_index(name="docs")
)

candidate_eval["total_docs"] = (
    candidate_eval
    .groupby("tnved_4")["docs"]
    .transform("sum")
)

candidate_eval["purity"] = (
    candidate_eval["docs"]
    / candidate_eval["total_docs"]
)

candidate_eval.sort_values(
    ["tnved_4", "purity"],
    ascending=[True, False]
)

,tnved_4,tg,docs,total_docs,purity
0,3306,35,990,991,0.998991
1,3306,4,1,991,0.001009
2,3307,35,3294,3324,0.990975
3,3307,4,29,3324,0.008724
4,3307,43,1,3324,0.000301
5,3402,35,1973,1994,0.989468
6,3402,43,21,1994,0.010532
7,3808,35,1055,1057,0.998108
8,3808,43,2,1057,0.001892


In [162]:
candidate_eval_coverage = (
    tg35_uncovered_tnved[
        tg35_uncovered_tnved["tnved_4"].isin(new_tnved_candidates)
    ]
    ["tnved_4"]
    .value_counts()
)

candidate_eval_coverage

tnved_4
3307    3230
3402    1973
3808    1055
3306     989
Name: count, dtype: int64

In [163]:
new_tg35_anchor_mask = (
    tg35_uncovered["tnved_4"].apply(
        lambda codes: any(
            code in new_tnved_candidates
            for code in codes
        )
    )
)

print("Новых TG35 документов, покрытых кандидатами:")
print(new_tg35_anchor_mask.sum())

print("Доля от uncovered:")
print(new_tg35_anchor_mask.mean())

print("Новая общая coverage TG35:")
print(
    (
        len(tg35_covered)
        + new_tg35_anchor_mask.sum()
    )
    / len(tg35_all)
)

Новых TG35 документов, покрытых кандидатами:
7239
Доля от uncovered:
0.378232927530174
Новая общая coverage TG35:
0.825947052800936


In [164]:
tg35_extended = (
    tg35_all["anchor_tg35"]
    |
    tg35_all["tnved_4"].apply(
        lambda codes: any(
            code in new_tnved_candidates
            for code in codes
        )
    )
)

print("Extended coverage TG35:")
print(tg35_extended.mean())

print("Extended uncovered:")
print((~tg35_extended).sum())

Extended coverage TG35:
0.825947052800936
Extended uncovered:
11900


In [165]:
new_definition_prefixes = [
    "3306",
    "3307",
    "3402",
    "3808",
]

definitions_clean[
    definitions_clean["tnved_4"].isin(new_definition_prefixes)
].sort_values(
    ["category", "tnved_4", "tnved_code"]
)

,category,tnved_code,tnved_name,tnved_4
12,Косметика,3306100000,3306100000 Средства для чистки зубов,3306
32,Косметика,3306200000,"3306200000 Нити, используемые для очистки межз...",3306
13,Косметика,3306900000,3306900000 Прочие средства для гигиены полости...,3306
14,Косметика,3307100000,"3307100000 Средства, используемые до, во время...",3307
15,Косметика,3307200000,3307200000 Дезодоранты и антиперспиранты индив...,3307
16,Косметика,3307300000,3307300000 Ароматизированные соли и прочие сос...,3307
17,Косметика,3307490000,3307490000 Прочие средства для ароматизации ил...,3307
18,Косметика,3307900008,3307900008 Косметические или туалетные средств...,3307
25,Косметика,3402500000,3402500000 Вещества поверхностно-активные орга...,3402


In [166]:
definitions_clean[
    definitions_clean["tnved_4"].isin(new_definition_prefixes)
].groupby(
    ["tnved_4", "category"]
)["tnved_code"].nunique()

tnved_4  category 
3306     Косметика    3
3307     Косметика    5
3402     Косметика    1
Name: tnved_code, dtype: int64

In [167]:
validated_new_tnved = [
    "3306",
    "3307",
    "3402",
]

tg35_extended_validated = (
    tg35_all["anchor_tg35"]
    |
    tg35_all["tnved_4"].apply(
        lambda codes: any(
            code in validated_new_tnved
            for code in codes
        )
    )
)

print(
    "TG35 coverage with domain-supported TNVED:",
    tg35_extended_validated.mean()
)

print(
    "Uncovered:",
    (~tg35_extended_validated).sum()
)

TG35 coverage with domain-supported TNVED: 0.8105455609185315
Uncovered: 12953


In [168]:
tg35_remaining = tg35_all[
    ~tg35_extended_validated
].copy()

In [169]:
tg35_remaining["tnved_4"].explode().value_counts().head(30)

tnved_4
3808    1053
8212      72
1905      60
3303      37
3301      31
2106      28
3405      24
1602      19
6104      13
0406      13
6110      11
1704      10
1902      10
6103       9
6106       9
1103       8
0403       8
1601       8
6204       7
6105       7
6112       7
2710       7
2008       7
1604       6
1806       6
2203       6
6206       6
3204       6
6211       5
2103       5
Name: count, dtype: int64

In [170]:
remaining_tnved_counts = (
    tg35_remaining[
        ["rd_documentnumber", "tnved_4"]
    ]
    .explode("tnved_4", ignore_index=True)
    .dropna(subset=["tnved_4"])
    .drop_duplicates(
        subset=["rd_documentnumber", "tnved_4"]
    )["tnved_4"]
    .value_counts()
)

remaining_tnved_counts.head(30)

tnved_4
3808    1053
8212      72
1905      60
3303      37
3301      31
2106      28
3405      24
1602      19
6104      13
0406      13
6110      11
1704      10
1902      10
6103       9
6106       9
1103       8
0403       8
1601       8
6204       7
6105       7
6112       7
2710       7
2008       7
1604       6
1806       6
2203       6
6206       6
3204       6
6211       5
2103       5
Name: count, dtype: int64

has_tnved_3808

Target: TG35

Support: ~1057

Purity: ~99.8%

Status: Strong Candidate

Domain validation: not found in TG Definitions

In [171]:
extended_with_3808 = (
    tg35_extended_validated
    |
    tg35_all["tnved_4"].apply(
        lambda codes: "3808" in codes
    )
)

print("TG35 coverage:", extended_with_3808.mean())
print("Uncovered:", (~extended_with_3808).sum())

TG35 coverage: 0.825947052800936
Uncovered: 11900


In [172]:
tg35_final_uncovered = tg35_all[
    ~extended_with_3808
].copy()

In [173]:
declaration_types = Counter()
declaration_keys = Counter()
declaration_nonempty = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    declaration = data.get("declaration")

    declaration_types[type(declaration).__name__] += 1

    if isinstance(declaration, dict):
        declaration_keys.update(declaration.keys())

        for key, value in declaration.items():
            if value not in ("", None, [], {}):
                declaration_nonempty[key] += 1

print("Типы declaration:")
print(declaration_types)

print("\nКлючи declaration:")
print(declaration_keys.most_common())

print("\nНепустые значения:")
print(declaration_nonempty.most_common())

Типы declaration:
Counter({'dict': 76435, 'NoneType': 9866})

Ключи declaration:
[('number', 76435), ('idDeclType', 76435), ('declRegDate', 76435), ('declEndDate', 76419), ('idDeclScheme', 76385), ('filialAddresses', 75477)]

Непустые значения:
[('number', 74642), ('idDeclType', 74642), ('declRegDate', 74642), ('declEndDate', 74593), ('idDeclScheme', 74580), ('filialAddresses', 73356)]


In [174]:
def extract_declaration_fields(row):
    data = json.loads(row)
    declaration = data.get("declaration")

    if not isinstance(declaration, dict):
        return pd.Series({
            "decl_scheme": None,
            "decl_type": None,
            "decl_reg_date": None,
            "decl_end_date": None,
            "decl_number": None,
        })

    return pd.Series({
        "decl_scheme": declaration.get("idDeclScheme"),
        "decl_type": declaration.get("idDeclType"),
        "decl_reg_date": declaration.get("declRegDate"),
        "decl_end_date": declaration.get("declEndDate"),
        "decl_number": declaration.get("number"),
    })

In [175]:
decl_features = base_df["rd_data"].apply(extract_declaration_fields)

base_df = pd.concat(
    [base_df, decl_features],
    axis=1
)

In [176]:
base_df[
    [
        "decl_scheme",
        "decl_type",
        "decl_reg_date",
        "decl_end_date",
        "decl_number"
    ]
].head()

,decl_scheme,decl_type,decl_reg_date,decl_end_date,decl_number
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,,,,,
4,,,,,


In [177]:
base_df["decl_scheme"].value_counts(dropna=False).head(20)

decl_scheme
3д                        54834
1д                        14202
NaN                        9916
6д                         3850
                           1805
Приложение № 18 ПП 353      878
2д                          611
4д                          175
7д                           16
Не определено                11
5д                            3
Name: count, dtype: int64

In [178]:
decl_tg = doc_tg_type.merge(
    base_df[
        [
            "rd_documentnumber",
            "decl_scheme",
            "decl_type",
            "decl_reg_date",
            "decl_end_date",
            "decl_number",
        ]
    ].rename(
        columns={"rd_documentnumber": "rd_number"}
    ),
    on="rd_number",
    how="inner"
)

In [179]:
print(decl_tg.shape)
print(decl_tg["rd_number"].nunique())

(86371, 8)
86301


In [180]:
decl_scheme_by_tg = pd.crosstab(
    decl_tg["tg"],
    decl_tg["decl_scheme"].fillna("Нет данных"),
    normalize="index"
)

decl_scheme_by_tg

decl_scheme,,1д,2д,3д,4д,5д,6д,7д,Не определено,Нет данных,Приложение № 18 ПП 353
tg,,,,,,,,,,,
35,0.025317,0.015854,0.002545,0.744742,0.001857,0.000044,0.055650,0.000234,0.000117,0.144046,0.009594
4,0.000254,0.001778,0.000000,0.960112,0.012195,0.000000,0.011687,0.000000,0.000000,0.002033,0.011941
43,0.005618,0.932935,0.031079,0.012659,0.000000,0.000000,0.000000,0.000000,0.000213,0.005049,0.012446


In [181]:
decl_scheme_precision = pd.crosstab(
    decl_tg["decl_scheme"].fillna("Нет данных"),
    decl_tg["tg"],
    normalize="index"
)

decl_scheme_precision

tg,35,4,43
decl_scheme,,,
,0.955826,0.000552,0.043622
1д,0.076290,0.000493,0.923218
2д,0.284779,0.000000,0.715221
3д,0.927895,0.068862,0.003244
4д,0.725714,0.274286,0.000000
5д,1.000000,0.000000,0.000000
6д,0.988055,0.011945,0.000000
7д,1.000000,0.000000,0.000000
Не определено,0.727273,0.000000,0.272727


In [182]:
decl_scheme_support = (
    decl_tg
    .groupby("decl_scheme")["rd_number"]
    .nunique()
    .sort_values(ascending=False)
)

decl_scheme_support

decl_scheme
3д                        54834
1д                        14202
6д                         3850
                           1805
Приложение № 18 ПП 353      878
2д                          611
4д                          175
7д                           16
Не определено                11
5д                            3
Name: rd_number, dtype: int64

In [183]:
decl_scheme_by_tg_type = pd.crosstab(
    [decl_tg["tg"], decl_tg["rd_type"]],
    decl_tg["decl_scheme"].fillna("Нет данных"),
    normalize="index"
)

decl_scheme_by_tg_type

decl_scheme                  1д        2д        3д        4д        5д  \
tg rd_type                                                                
35 ДС        0.000158  0.019071  0.003061  0.895797  0.002234  0.000053   
   СГР       0.150950  0.000000  0.000000  0.000000  0.000000  0.000000   
   СС        0.081181  0.000000  0.000000  0.011070  0.000000  0.000000   
4  N/A       0.000254  0.001778  0.000000  0.960112  0.012195  0.000000   
43 ДС        0.000144  0.942859  0.031409  0.012794  0.000000  0.000000   
   СГР       0.468254  0.000000  0.000000  0.000000  0.000000  0.000000   
   СС        0.818182  0.000000  0.000000  0.000000  0.000000  0.000000   

decl_scheme        6д        7д  Не определено  Нет данных  \
tg rd_type                                                   
35 ДС        0.066924  0.000281       0.000141    0.000739   
   СГР       0.000000  0.000000       0.000000    0.849050   
   СС        0.003690  0.000000       0.000000    0.904059   
4  N/A       0.011687  0.000000       0.000000    0.002033   
43 ДС        0.000000  0.000000       0.000216    0.000000   
   СГР       0.000000  0.000000       0.000000    0.531746   
   СС        0.000000  0.000000       0.000000    0.181818   

decl_scheme  Приложение № 18 ПП 353  
tg rd_type                           
35 ДС                      0.011541  
   СГР                     0.000000  
   СС                      0.000000  
4  N/A                     0.011941  
43 ДС                      0.012578  
   СГР                     0.000000  
   СС                      0.000000

In [184]:
base_df["decl_scheme_clean"] = (
    base_df["decl_scheme"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

In [185]:
tg35_final_uncovered = tg35_all[
    ~extended_with_3808
].copy()

print("Оставшиеся TG35:", len(tg35_final_uncovered))

Оставшиеся TG35: 11900


In [186]:
decl_lookup = (
    base_df[
        ["rd_documentnumber", "decl_scheme"]
    ]
    .drop_duplicates("rd_documentnumber")
)

tg35_final_uncovered = tg35_final_uncovered.merge(
    decl_lookup,
    left_on="rd_documentnumber",
    right_on="rd_documentnumber",
    how="left",
    validate="many_to_one"
)

In [187]:
print(tg35_final_uncovered.shape)
print(tg35_final_uncovered["decl_scheme"].notna().mean())

(11900, 45)
0.19563025210084034


In [188]:
tg35_final_uncovered["decl_scheme_clean"] = (
    tg35_final_uncovered["decl_scheme"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

In [189]:
tg35_final_uncovered["decl_scheme_clean"].value_counts(
    dropna=False
)

decl_scheme_clean
<NA>                      11276
3д                          445
1д                          140
2д                           12
6д                           11
7д                            5
Приложение № 18 ПП 353        4
4д                            4
5д                            3
Name: count, dtype: int64[pyarrow]

In [190]:
scheme_candidates = [
    "1д",
    "2д",
    "3д",
    "6д",
    "4д"
]

tg35_final_uncovered[
    "decl_scheme_clean"
].isin(scheme_candidates).mean()

np.float64(0.05142857142857143)

In [191]:
tg35_final_uncovered = tg35_final_uncovered.reset_index(drop=True)

In [192]:
tg35_final_uncovered["tech_regulations"].value_counts(
    dropna=False
).head(15)

tech_regulations
NaN                                                                                                                                                                                                                                                                                     11414
ТР ТС 009/2011 О безопасности парфюмерно-косметической продукции                                                                                                                                                                                                                          128
ТР ТС 022/2011 Пищевая продукция в части ее маркировки, ТР ТС 021/2011 О безопасности пищевой продукции, ТР ТС 029/2012 Требования безопасности пищевых добавок, ароматизаторов и технологических вспомогательных средств                                                                  64
ТР ТС 017/2011 О безопасности продукции легкой промышленности                                                                

In [193]:
tg35_final_uncovered[
    "tech_regulations"
].notna().mean()

np.float64(0.04084033613445378)

In [194]:
tg35_final_uncovered["has_tr009"] = (
    tg35_final_uncovered["tech_reg_codes"]
    .apply(lambda x: "ТР ТС 009/2011" in x)
)

tg35_final_uncovered["has_tr030"] = (
    tg35_final_uncovered["tech_reg_codes"]
    .apply(lambda x: "ТР ТС 030/2012" in x)
)

print(
    "TR009:",
    tg35_final_uncovered["has_tr009"].mean()
)

print(
    "TR030:",
    tg35_final_uncovered["has_tr030"].mean()
)

TR009: 0.010756302521008404
TR030: 0.0009243697478991597


In [195]:
residual_presence = pd.DataFrame({
    "tech_regulations": tg35_final_uncovered["tech_reg_codes"].apply(len).gt(0),
    "manufacturer_type": tg35_final_uncovered["manufacturer_type"].notna(),
    "applicant_type": tg35_final_uncovered["applicant_type"].notna(),
    "product_origin": tg35_final_uncovered["product_origin"].notna(),
    "product_object_type": tg35_final_uncovered["product_object_type"].notna(),
    "decl_scheme": tg35_final_uncovered["decl_scheme_clean"].notna(),
})

residual_presence.mean().sort_values(ascending=False)

product_object_type    0.053613
manufacturer_type      0.052689
applicant_type         0.052689
decl_scheme            0.052437
product_origin         0.047647
tech_regulations       0.040840
dtype: float64

In [196]:
residual_docs = tg35_final_uncovered[
    ["rd_documentnumber"]
].drop_duplicates()

residual_fresh = residual_docs.merge(
    base_df[
        [
            "rd_documentnumber",
            "manufacturer_type",
            "applicant_type",
            "product_object_type",
            "product_origin",
            "tech_reg_codes",
        ]
    ].drop_duplicates("rd_documentnumber"),
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

residual_fresh.shape

(11900, 6)

In [197]:
residual_presence = pd.DataFrame({
    "coverage": [
        residual_fresh["manufacturer_type"].notna().mean(),
        residual_fresh["applicant_type"].notna().mean(),
        residual_fresh["product_object_type"].notna().mean(),
        residual_fresh["product_origin"].notna().mean(),
        residual_fresh["tech_reg_codes"].apply(len).gt(0).mean(),
    ]
}, index=[
    "manufacturer_type",
    "applicant_type",
    "product_object_type",
    "product_origin",
    "tech_regulations",
])

residual_presence

,coverage
manufacturer_type,0.052689
applicant_type,0.052689
product_object_type,0.053613
product_origin,0.047647
tech_regulations,0.040840


In [198]:
print(
    "product_name coverage:",
    tg35_final_uncovered["product_name"].notna().mean()
)

print(
    "non-empty:",
    tg35_final_uncovered["product_name"]
    .fillna("")
    .str.strip()
    .ne("")
    .mean()
)

product_name coverage: 0.05361344537815126
non-empty: 0.05361344537815126


In [199]:
tg35_final_uncovered.loc[
    tg35_final_uncovered["product_name"]
    .fillna("")
    .str.strip()
    .ne(""),
    [
        "rd_documentnumber",
        "product_name",
        "product_info"
    ]
].sample(
    50,
    random_state=42
)

,rd_documentnumber,product_name,product_info
11522,ЕАЭС N RU Д-RU.РА01.В.47878/24,Продукция парфюмерная жидкая: Духи торговой ма...,;
11490,ЕАЭС N RU Д-RU.РА01.В.40322/21,"Кефир, мацони, простокваша, йогурт питьевой кл...",NaN
11293,ЕАЭС N RU Д-CN.РА10.В.39297/24,Обувь домашняя с верхом из текстильных материа...,"туфли комнатные (тапочки), мюли домашние, ..."
11845,РОСС RU Д-RU.РА01.В.14420/24,Товары бытовой химии:,Свидетельства о государственной регистрац...
11862,РОСС RU Д-RU.РА01.В.22166/24,Средство универсальное для стирки,"OXI MOXI, Свидетельство о государственной ..."
11292,ЕАЭС N RU Д-CN.РА10.В.13275/24,"Средства косметические для макияжа губ, в том ...","марка: COSMISO есо professional, ZOZU, Guangz..."
11855,РОСС RU Д-RU.РА01.В.19619/24,Одноразовый бритвенный станок,"торговой марки Cantlay,"
11372,ЕАЭС N RU Д-RU.РА01.В.12291/21,Сухая смесь для изготовления кондитерского кру...,NaN
11509,ЕАЭС N RU Д-RU.РА01.В.43722/21,Концентраты пищевые сладких блюд. Кисель сухой...,NaN
11785,РОСС RU Д-CN.РА01.В.34955/25,Предметы металлической галантереи: станок брит...,"торговая марка: WETELL, Договор уполномоче..."


In [200]:
print(
    "product_info coverage:",
    tg35_final_uncovered["product_info"].notna().mean()
)

print(
    "non-empty:",
    tg35_final_uncovered["product_info"]
    .fillna("")
    .str.strip()
    .ne("")
    .mean()
)

product_info coverage: 0.031932773109243695
non-empty: 0.02


In [201]:
tg35_final_uncovered.loc[
    tg35_final_uncovered["product_info"]
    .fillna("")
    .str.strip()
    .ne(""),
    [
        "rd_documentnumber",
        "product_name",
        "product_info"
    ]
].sample(
    50,
    random_state=42
)

,rd_documentnumber,product_name,product_info
11687,ЕАЭС N RU Д-RU.РА04.В.77966/22,Хлебобулочные изделия в упаковке и без упаковк...,"пирожки печеные с мясом, луком и рисом; пирожк..."
11297,ЕАЭС N RU Д-DE.РА01.В.43567/21,Инструменты и машины ручные электрические,"согласно приложению № 1 на 1 листе, , ; ..."
11849,РОСС RU Д-RU.РА01.В.16318/25,Набор мужских одноразовых бритвенных станков с...,"торговая марка RAPIRA,"
11699,ЕАЭС N RU Д-RU.РА05.В.53829/22,Продукция косметическая для ухода за кожей,"ARKADIA, Упаковка: банки, флаконы, тубы, а..."
11283,ЕАЭС N RU Д-CN.РА01.В.92653/24,"Обувь с подошвой из резины, пластмассы, натура...",Декларация соответствия распространяется ...
11761,ЕАЭС RU С-RU.АЯ24.В.00643/21,Изделия чулочно-носочные трикотажные первого с...,"с маркировкой «Gde noski», «Fabrikanoskov.ru»,..."
11289,ЕАЭС N RU Д-CN.РА08.В.14030/23,Изделия 3-го слоя верхние трикотажные для мужч...,"befree, BEFREE MAN, LOVE REPUBLIC, SELA, moms..."
11862,РОСС RU Д-RU.РА01.В.22166/24,Средство универсальное для стирки,"OXI MOXI, Свидетельство о государственной ..."
11684,ЕАЭС N RU Д-RU.РА04.В.29273/25,Вода минеральная природная столовая питьевая,Упаковка - стеклянные бутылки различной в...
11861,РОСС RU Д-RU.РА01.В.20651/25,"Товары бытовой химии: Средство для удаления жира,","с маркировкой «Хороший помощник», «Голубая лу..."


In [202]:
case_rank1 = case_df[
    case_df["rank"] == 1
][
    ["rd_documentnumber", "tg_ids"]
].copy()

print(case_rank1.shape)

(3595803, 2)


In [203]:
tg_ids_lookup = (
    case_rank1
    .drop_duplicates("rd_documentnumber")
)

tg35_residual_check = tg35_final_uncovered[
    ["rd_documentnumber", "tg"]
].drop_duplicates()

tg35_residual_check = tg35_residual_check.merge(
    tg_ids_lookup,
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

print(tg35_residual_check.shape)

(11900, 3)


In [204]:
tg35_residual_check["tg_ids"].head(20)

0         [35]
1         [35]
2         [35]
3     [35, 43]
4     [35, 43]
5     [35, 43]
6     [35, 43]
7      [35, 4]
8      [35, 4]
9      [35, 4]
10     [35, 4]
11     [35, 4]
12     [35, 4]
13     [35, 4]
14     [35, 4]
15     [35, 4]
16     [35, 4]
17     [35, 4]
18        [35]
19     [35, 4]
Name: tg_ids, dtype: object

In [205]:
tg35_residual_check["tg_ids"].apply(
    lambda x: str(x)
).value_counts().head(30)

tg_ids
[35]             7457
[35  4]          2383
[35 37]           573
[35  5]           245
[35  4 37]        211
[ 1 35]           162
[37]              102
[35 43]            88
[45]               80
[1]                65
[35 45]            50
[35  4 45]         44
[8]                35
[35  4 43]         33
[ 2 35]            32
[35  4 37 45]      28
[35  5 37]         23
[37 35 45]         22
[43]               22
[ 8 35]            21
[35 37 45]         21
[4]                18
[45 37]            18
[5]                15
[43 35]            12
[]                 10
[2]                 8
[ 8 35 37]          7
[35  5 37  8]       7
[ 1 35 37]          6
Name: count, dtype: int64

In [206]:
tg35_residual_check["truth_tg_str"] = (
    tg35_residual_check["tg"]
    .apply(lambda x: str(x))
)

tg35_residual_check["tg_ids_str"] = (
    tg35_residual_check["tg_ids"]
    .apply(lambda x: str(x))
)

tg35_residual_check[
    ["truth_tg_str", "tg_ids_str"]
].value_counts().head(30)

truth_tg_str  tg_ids_str   
35            [35]             7457
              [35  4]          2383
              [35 37]           573
              [35  5]           245
              [35  4 37]        211
              [ 1 35]           162
              [37]              102
              [35 43]            88
              [45]               80
              [1]                65
              [35 45]            50
              [35  4 45]         44
              [8]                35
              [35  4 43]         33
              [ 2 35]            32
              [35  4 37 45]      28
              [35  5 37]         23
              [37 35 45]         22
              [43]               22
              [ 8 35]            21
              [35 37 45]         21
              [4]                18
              [45 37]            18
              [5]                15
              [43 35]            12
              []                 10
              [2]                 8


In [207]:
tg35_residual_check[
    ~tg35_residual_check["tg_ids_str"].isin(
        ["[35]", "35"]
    )
].head(50)

,rd_documentnumber,tg,tg_ids,truth_tg_str,tg_ids_str
3,AM.02.01.01.001.R.000076.12.20,35,"[35, 43]",35,[35 43]
4,AM.02.01.01.001.R.000077.12.20,35,"[35, 43]",35,[35 43]
5,AM.02.01.01.001.R.000078.12.20,35,"[35, 43]",35,[35 43]
6,AM.02.01.01.001.R.000079.12.20,35,"[35, 43]",35,[35 43]
7,AM.02.01.01.001.R.000393.08.22,35,"[35, 4]",35,[35 4]
8,AM.02.07.01.001.R.000004.01.23,35,"[35, 4]",35,[35 4]
9,AM.02.07.01.001.R.000005.01.23,35,"[35, 4]",35,[35 4]
10,AM.02.07.01.001.R.000007.01.22,35,"[35, 4]",35,[35 4]
11,AM.02.07.01.001.R.000008.01.22,35,"[35, 4]",35,[35 4]
12,AM.02.07.01.001.R.000011.01.23,35,"[35, 4]",35,[35 4]


In [208]:
def tg_set(x):
    if x is None:
        return set()
    
    try:
        return set(int(v) for v in x)
    except (TypeError, ValueError):
        return set()


tg35_residual_check["tg_set"] = (
    tg35_residual_check["tg_ids"]
    .apply(tg_set)
)

tg35_residual_check["has_35"] = (
    tg35_residual_check["tg_set"]
    .apply(lambda x: 35 in x)
)

tg35_residual_check["has_target_other"] = (
    tg35_residual_check["tg_set"]
    .apply(lambda x: bool(x & {4, 43}))
)

tg35_residual_check["has_non_target"] = (
    tg35_residual_check["tg_set"]
    .apply(lambda x: bool(x - {4, 35, 43}))
)

In [209]:
pd.Series({
    "has_35": tg35_residual_check["has_35"].mean(),
    "no_35": (~tg35_residual_check["has_35"]).mean(),
    "has_4_or_43": tg35_residual_check["has_target_other"].mean(),
    "has_non_target": tg35_residual_check["has_non_target"].mean(),
})

has_35            0.965462
no_35             0.034538
has_4_or_43       0.241176
has_non_target    0.157731
dtype: float64

In [210]:
pd.crosstab(
    tg35_residual_check["has_35"],
    tg35_residual_check["has_target_other"]
)

has_target_other,False,True
has_35,,
False,369,42
True,8661,2828


In [211]:
def get_field(row, field):
    data = json.loads(row)
    value = data.get(field)

    if isinstance(value, str):
        value = value.strip()

    return value

In [212]:
tg35_final_uncovered["nameProd"] = (
    tg35_final_uncovered["rd_data"]
    .apply(lambda x: get_field(x, "nameProd"))
)

In [213]:
print(
    "nameProd coverage:",
    tg35_final_uncovered["nameProd"].notna().mean()
)

print(
    "nameProd non-empty:",
    tg35_final_uncovered["nameProd"]
    .fillna("")
    .str.strip()
    .ne("")
    .mean()
)

nameProd coverage: 0.9770588235294118
nameProd non-empty: 0.9456302521008403


In [214]:
tg35_final_uncovered.loc[
    tg35_final_uncovered["nameProd"]
    .fillna("")
    .str.strip()
    .ne(""),
    [
        "rd_documentnumber",
        "nameProd"
    ]
].sample(
    50,
    random_state=42
)

,rd_documentnumber,nameProd
7419,RU.77.01.34.015.Е.000052.01.24,Средство для чистки поверхностей CA 30 C eco!p...
6094,RU.66.01.40.015.Е.000100.07.21,"Средства чистящие универсальные: ""Sanita""® (ил..."
1440,KG.11.01.09.001.R.001718.05.20,"Пилинг-скатка гиалуроновая товарного знака ""NO..."
10882,RU.78.01.08.015.Е.000189.08.24,Товары бытовой химии: Концентрированный ополас...
8326,RU.77.01.34.015.Е.014069.09.11,Стиральный порошок торговой марки Lion (Лион)
3559,RU.08.08.09.015.Е.002945.11.23,Таблетки для посудомоечной машины CAPYFRESH.
2903,RU.01.РА.02.015.Е.000752.06.21,Чистящее средство для кофемашин: Cup 3 жидкое ...
4838,RU.50.99.05.001.R.000266.11.22,"продукция косметическая, произведенная с испол..."
7261,RU.77.01.34.001.Е.003338.11.17,Средства косметические для детей: Шампунь ДЕТС...
6076,RU.66.01.40.015.Е.000083.06.20,Экогель для чистки сантехники WONDER LAB


In [215]:
import re

def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[^а-яёa-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


tg35_final_uncovered["nameProd_clean"] = (
    tg35_final_uncovered["nameProd"]
    .apply(normalize_text)
)

In [216]:
from collections import Counter

token_counter = Counter()

for text in tg35_final_uncovered["nameProd_clean"]:
    token_counter.update(text.split())

token_counter.most_common(100)

[('для', 11773),
 ('средство', 4958),
 ('с', 3157),
 ('и', 3097),
 ('продукция', 2522),
 ('косметическая', 2484),
 ('в', 2199),
 ('марки', 2011),
 ('волос', 1675),
 ('гель', 1602),
 ('6', 1495),
 ('крем', 1493),
 ('7', 1413),
 ('5', 1393),
 ('средства', 1321),
 ('1', 1276),
 ('0', 1270),
 ('серии', 1174),
 ('мытья', 1152),
 ('8', 1138),
 ('9', 1103),
 ('детей', 1009),
 ('10', 988),
 ('3', 975),
 ('4', 946),
 ('моющее', 922),
 ('посуды', 857),
 ('дезинфицирующее', 827),
 ('торговой', 821),
 ('чистящее', 795),
 ('стирки', 792),
 ('том', 733),
 ('числе', 733),
 ('т', 710),
 ('детский', 702),
 ('м', 671),
 ('за', 649),
 ('белья', 633),
 ('2', 630),
 ('тона', 597),
 ('краска', 589),
 ('жидкое', 575),
 ('бытовой', 545),
 ('color', 543),
 ('химии', 538),
 ('маркировкой', 531),
 ('ухода', 518),
 ('очиститель', 515),
 ('наборах', 514),
 ('окрашивания', 507),
 ('гигиены', 502),
 ('профессионального', 493),
 ('12', 491),
 ('professional', 470),
 ('мыло', 469),
 ('шампунь', 461),
 ('знака', 461),


In [217]:
stop_words = {
    "и", "в", "во", "на", "с", "со", "для", "по", "из", "к", "у",
    "о", "об", "от", "до", "не", "а",
    "продукция", "продукт", "средство", "товар", "изделие",
}

In [218]:
filtered_counter = Counter()

for text in tg35_final_uncovered["nameProd_clean"]:
    tokens = [
        token
        for token in text.split()
        if token not in stop_words
        and len(token) >= 3
    ]

    filtered_counter.update(tokens)

filtered_counter.most_common(100)

[('косметическая', 2484),
 ('марки', 2011),
 ('волос', 1675),
 ('гель', 1602),
 ('крем', 1493),
 ('средства', 1321),
 ('серии', 1174),
 ('мытья', 1152),
 ('детей', 1009),
 ('моющее', 922),
 ('посуды', 857),
 ('дезинфицирующее', 827),
 ('торговой', 821),
 ('чистящее', 795),
 ('стирки', 792),
 ('том', 733),
 ('числе', 733),
 ('детский', 702),
 ('белья', 633),
 ('тона', 597),
 ('краска', 589),
 ('жидкое', 575),
 ('бытовой', 545),
 ('color', 543),
 ('химии', 538),
 ('маркировкой', 531),
 ('ухода', 518),
 ('очиститель', 515),
 ('наборах', 514),
 ('окрашивания', 507),
 ('гигиены', 502),
 ('профессионального', 493),
 ('professional', 470),
 ('мыло', 469),
 ('шампунь', 461),
 ('знака', 461),
 ('приложению', 455),
 ('товарного', 450),
 ('далее', 448),
 ('согласно', 446),
 ('универсальное', 445),
 ('детская', 443),
 ('детское', 426),
 ('товары', 413),
 ('пилинг', 409),
 ('интимной', 405),
 ('кондиционер', 400),
 ('косметическое', 391),
 ('кожей', 390),
 ('спрей', 369),
 ('применения', 362),
 ('к

In [219]:
def extract_name_prod(row):
    data = json.loads(row)
    value = data.get("nameProd")

    if isinstance(value, str):
        value = value.strip()
        return value if value else None

    return None


base_df["nameProd"] = base_df["rd_data"].apply(extract_name_prod)

In [220]:
print(base_df["nameProd"].notna().mean())

0.13165548487271295


In [221]:
text_eval = (
    base_df[
        ["rd_documentnumber", "tg", "nameProd"]
    ]
    .explode("tg", ignore_index=True)
    .copy()
)

text_eval["tg"] = text_eval["tg"].astype("string")

text_eval["nameProd_clean"] = (
    text_eval["nameProd"]
    .apply(normalize_text)
)

text_eval = text_eval[
    text_eval["nameProd_clean"].ne("")
].copy()

print(text_eval.shape)
print(text_eval["tg"].value_counts())

(11379, 4)
tg
35    11253
43      126
Name: count, dtype: int64[pyarrow]


In [222]:
term_docs = []

for tg in ["4", "35", "43"]:
    tg_docs = text_eval[text_eval["tg"] == tg]

    for rd_number, text in zip(
        tg_docs["rd_documentnumber"],
        tg_docs["nameProd_clean"]
    ):
        terms = set(
            token
            for token in text.split()
            if token not in stop_words
            and len(token) >= 3
        )

        for term in terms:
            term_docs.append((term, tg, rd_number))

term_docs = pd.DataFrame(
    term_docs,
    columns=["term", "tg", "rd_number"]
).drop_duplicates()

In [223]:
term_support = (
    term_docs
    .groupby(["term", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
)

term_support.head()

tg,35,43
term,,
000,10,0
0000,2,0
00000173,1,0
0001,9,0
0002,9,0


In [224]:
term_support["total_docs"] = (
    term_support.sum(axis=1)
)

In [225]:
term_support["tg35_precision"] = (
    term_support.get("35", 0)
    / term_support["total_docs"]
)

In [226]:
term_support["tg4_precision"] = (
    term_support.get("4", 0)
    / term_support["total_docs"]
)

term_support["tg43_precision"] = (
    term_support.get("43", 0)
    / term_support["total_docs"]
)

In [227]:
tg35_term_candidates = (
    term_support[
        term_support["35"] >= 100
    ]
    .sort_values(
        ["tg35_precision", "35"],
        ascending=[False, False]
    )
)

tg35_term_candidates.head(50)

tg,35,43,total_docs,tg35_precision,tg4_precision,tg43_precision
term,,,,,,
мытья,847,0,847,1.0,0.0,0.0
том,678,0,678,1.0,0.0,0.0
числе,678,0,678,1.0,0.0,0.0
чистящее,611,0,611,1.0,0.0,0.0
посуды,596,0,596,1.0,0.0,0.0
детский,538,0,538,1.0,0.0,0.0
краска,514,0,514,1.0,0.0,0.0
ухода,501,0,501,1.0,0.0,0.0
наборах,490,0,490,1.0,0.0,0.0


In [228]:
nameprod_by_tg = (
    base_df
    .explode("tg")
    .assign(tg=lambda x: x["tg"].astype("string"))
)

nameprod_by_tg["nameProd_nonempty"] = (
    nameprod_by_tg["nameProd"]
    .fillna("")
    .str.strip()
    .ne("")
)

nameprod_coverage = (
    nameprod_by_tg
    .groupby("tg")["nameProd_nonempty"]
    .agg(
        total_docs="count",
        docs_with_nameProd="sum",
        coverage="mean"
    )
)

nameprod_coverage

,total_docs,docs_with_nameProd,coverage
tg,,,
35,68370,11253,0.164590
4,3936,0,0.000000
43,14061,126,0.008961


In [229]:
term_support = (
    term_docs
    .groupby(["term", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["4", "35", "43"], fill_value=0)
)

term_support["total_docs"] = term_support.sum(axis=1)

term_support["tg35_precision"] = (
    term_support["35"] / term_support["total_docs"]
)

term_support["tg35_recall"] = (
    term_support["35"] / 
    nameprod_coverage.loc["35", "docs_with_nameProd"]
)

In [230]:
term_support["tg35_global_recall"] = (
    term_support["35"] / 
    nameprod_coverage.loc["35", "total_docs"]
)

In [231]:
tg35_term_candidates = (
    term_support[
        (term_support["35"] >= 100)
        & (term_support["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["tg35_global_recall", "35"],
        ascending=[False, False]
    )
)

tg35_term_candidates.head(50)

tg,4,35,43,total_docs,tg35_precision,tg35_recall,tg35_global_recall
term,,,,,,,
косметическая,0,2484,3,2487,0.998794,0.220741,0.036332
марки,0,1895,14,1909,0.992666,0.168400,0.027717
средства,0,1293,19,1312,0.985518,0.114903,0.018912
крем,0,1228,5,1233,0.995945,0.109126,0.017961
волос,0,1205,1,1206,0.999171,0.107083,0.017625
гель,0,1166,1,1167,0.999143,0.103617,0.017054
детей,0,917,3,920,0.996739,0.081489,0.013412
серии,0,904,2,906,0.997792,0.080334,0.013222
мытья,0,847,0,847,1.000000,0.075269,0.012388


In [232]:
residual_term_support = (
    term_docs[
        term_docs["tg"] == "35"
    ]
    .groupby("term")["rd_number"]
    .nunique()
    .sort_values(ascending=False)
)

residual_term_support.head(50)

term
косметическая        2484
марки                1895
средства             1293
крем                 1228
волос                1205
гель                 1166
детей                 917
серии                 904
мытья                 847
дезинфицирующее       814
моющее                805
торговой              752
числе                 678
том                   678
чистящее              611
посуды                596
тона                  577
детский               538
бытовой               531
химии                 525
краска                514
ухода                 501
наборах               490
маркировкой           482
окрашивания           482
стирки                481
профессионального     480
жидкое                466
приложению            454
далее                 448
белья                 445
согласно              444
знака                 437
гигиены               432
товарного             426
товары                413
professional          411
кожей                 386
космети

In [233]:
from collections import Counter

bigram_docs = []

for tg, rd_number, text in zip(
    text_eval["tg"],
    text_eval["rd_documentnumber"],
    text_eval["nameProd_clean"]
):
    tokens = [
        token
        for token in text.split()
        if token not in stop_words
        and len(token) >= 3
    ]

    bigrams = set(
        zip(tokens, tokens[1:])
    )

    for a, b in bigrams:
        bigram_docs.append(
            (f"{a} {b}", tg, rd_number)
        )

bigram_docs = (
    pd.DataFrame(
        bigram_docs,
        columns=["bigram", "tg", "rd_number"]
    )
    .drop_duplicates()
)

In [234]:
bigram_support = (
    bigram_docs
    .groupby(["bigram", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["4", "35", "43"], fill_value=0)
)

bigram_support["total_docs"] = (
    bigram_support.sum(axis=1)
)

bigram_support["tg35_precision"] = (
    bigram_support["35"]
    / bigram_support["total_docs"]
)

bigram_support["tg35_global_recall"] = (
    bigram_support["35"]
    / nameprod_coverage.loc["35", "total_docs"]
)

In [235]:
bigram_candidates = (
    bigram_support[
        (bigram_support["35"] >= 30)
        & (bigram_support["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["tg35_precision", "35"],
        ascending=[False, False]
    )
)

bigram_candidates.head(50)

tg,4,35,43,total_docs,tg35_precision,tg35_global_recall
bigram,,,,,,
том числе,0,678,0,678,1.0,0.009917
мытья посуды,0,511,0,511,1.0,0.007474
окрашивания волос,0,452,0,452,1.0,0.006611
числе наборах,0,413,0,413,1.0,0.006041
краска волос,0,346,0,346,1.0,0.005061
крем краска,0,341,0,341,1.0,0.004988
интимной гигиены,0,336,0,336,1.0,0.004914
косметическая окрашивания,0,296,0,296,1.0,0.004329
ухода кожей,0,293,0,293,1.0,0.004286


In [236]:
tg4_sample = base_df[
    base_df["tg"].apply(lambda x: 4 in x)
][
    ["rd_documentnumber", "rd_data"]
].head(10)

tg4_sample

,rd_documentnumber,rd_data


In [237]:
for row in tg4_sample["rd_data"]:
    data = json.loads(row)
    print(data.get("nameProd"))

In [238]:
text_stop_words = stop_words | {
    "том",
    "числе",
    "марки",
    "серии",
    "торговой",
    "торговый",
    "торговая",
    "знака",
    "товарного",
    "приложению",
    "далее",
    "согласно",
    "применения",
    "использованием",
    "использования",
    "произведенная",
    "произведено",
    "профессионального",
    "профессиональной",
}

In [239]:
bigram_docs_clean = []

for tg, rd_number, text in zip(
    text_eval["tg"],
    text_eval["rd_documentnumber"],
    text_eval["nameProd_clean"]
):
    tokens = [
        token
        for token in text.split()
        if token not in text_stop_words
        and len(token) >= 3
        and not token.isdigit()
    ]

    bigrams = set(zip(tokens, tokens[1:]))

    for a, b in bigrams:
        bigram_docs_clean.append(
            (f"{a} {b}", tg, rd_number)
        )

bigram_docs_clean = (
    pd.DataFrame(
        bigram_docs_clean,
        columns=["bigram", "tg", "rd_number"]
    )
    .drop_duplicates()
)

In [240]:
bigram_support_clean = (
    bigram_docs_clean
    .groupby(["bigram", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
)

bigram_support_clean["total_docs"] = (
    bigram_support_clean.sum(axis=1)
)

bigram_support_clean["tg35_precision"] = (
    bigram_support_clean["35"]
    / bigram_support_clean["total_docs"]
)

In [241]:
print(bigram_support_clean.columns)

Index(['35', '43', 'total_docs', 'tg35_precision'], dtype='str', name='tg')


In [242]:
bigram_support_clean["tg35_support"] = (
    bigram_support_clean["35"]
)

bigram_support_clean["tg35_global_recall"] = (
    bigram_support_clean["35"]
    / nameprod_coverage.loc["35", "total_docs"]
)

bigram_support_clean["tg35_nameprod_recall"] = (
    bigram_support_clean["35"]
    / nameprod_coverage.loc["35", "docs_with_nameProd"]
)

In [243]:
bigram_candidates_clean = (
    bigram_support_clean[
        (bigram_support_clean["35"] >= 50)
        & (bigram_support_clean["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["tg35_support", "tg35_precision"],
        ascending=[False, False]
    )
)

bigram_candidates_clean.head(50)

tg,35,43,total_docs,tg35_precision,tg35_support,tg35_global_recall,tg35_nameprod_recall
bigram,,,,,,,
мытья посуды,511,0,511,1.000000,511,0.007474,0.045410
бытовой химии,501,11,512,0.978516,501,0.007328,0.044521
окрашивания волос,452,0,452,1.000000,452,0.006611,0.040167
товары бытовой,413,3,416,0.992788,413,0.006041,0.036701
косметическая детей,373,2,375,0.994667,373,0.005456,0.033147
косметическая окрашивания,360,0,360,1.000000,360,0.005265,0.031991
краска волос,346,0,346,1.000000,346,0.005061,0.030747
крем краска,341,0,341,1.000000,341,0.004988,0.030303
интимной гигиены,336,0,336,1.000000,336,0.004914,0.029859


In [244]:
residual_text_eval = (
    tg35_final_uncovered[
        ["rd_documentnumber", "nameProd_clean"]
    ]
    .drop_duplicates("rd_documentnumber")
)

residual_text_eval = residual_text_eval[
    residual_text_eval["nameProd_clean"].ne("")
].copy()

In [245]:
residual_bigram_support = Counter()

for text in residual_text_eval["nameProd_clean"]:
    tokens = [
        token
        for token in text.split()
        if token not in text_stop_words
        and len(token) >= 3
        and not token.isdigit()
    ]

    bigrams = set(zip(tokens, tokens[1:]))

    for a, b in bigrams:
        residual_bigram_support[f"{a} {b}"] += 1

residual_bigram_support = (
    pd.Series(residual_bigram_support, name="residual_docs")
    .sort_values(ascending=False)
)

residual_bigram_support.head(50)

мытья посуды                    511
бытовой химии                   501
окрашивания волос               452
товары бытовой                  413
косметическая детей             373
косметическая окрашивания       360
краска волос                    346
крем краска                     341
интимной гигиены                336
ухода кожей                     293
косметическая ухода             260
торговых марок                  161
детей взрослых                  159
кондиционер белья               156
гель интимной                   154
гигиеническая моющая            153
зубная паста                    153
косметическая гигиеническая     151
средства моющие                 142
посудомоечных машин             141
моющие средства                 137
косметическая наноматериалов    124
гель стирки                     122
средства косметические          121
средства чистящие               119
полости рта                     106
средства мытья                  103
парфюмерно косметическая    

In [246]:
bigram_docs_clean = []

for tg, rd_number, text in zip(
    text_eval["tg"],
    text_eval["rd_documentnumber"],
    text_eval["nameProd_clean"]
):
    tokens = [
        token
        for token in text.split()
        if len(token) >= 3
        and not token.isdigit()
    ]

    bigrams = set(zip(tokens, tokens[1:]))

    for a, b in bigrams:
        if a in text_stop_words or b in text_stop_words:
            continue

        bigram_docs_clean.append(
            (f"{a} {b}", tg, rd_number)
        )

bigram_docs_clean = (
    pd.DataFrame(
        bigram_docs_clean,
        columns=["bigram", "tg", "rd_number"]
    )
    .drop_duplicates()
)

In [247]:
bigram_support_clean = (
    bigram_docs_clean
    .groupby(["bigram", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["35", "43"], fill_value=0)
)

bigram_support_clean["total_docs"] = (
    bigram_support_clean["35"]
    + bigram_support_clean["43"]
)

bigram_support_clean["tg35_precision"] = (
    bigram_support_clean["35"]
    / bigram_support_clean["total_docs"]
)

bigram_support_clean["tg35_global_recall"] = (
    bigram_support_clean["35"]
    / nameprod_coverage.loc["35", "total_docs"]
)

In [248]:
bigram_candidates_clean = (
    bigram_support_clean[
        (bigram_support_clean["35"] >= 50)
        & (bigram_support_clean["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["35", "tg35_precision"],
        ascending=[False, False]
    )
)

bigram_candidates_clean.head(50)

tg,35,43,total_docs,tg35_precision,tg35_global_recall
bigram,,,,,
мытья посуды,511,0,511,1.000000,0.007474
бытовой химии,501,11,512,0.978516,0.007328
окрашивания волос,452,0,452,1.000000,0.006611
товары бытовой,413,3,416,0.992788,0.006041
крем краска,341,0,341,1.000000,0.004988
интимной гигиены,336,0,336,1.000000,0.004914
ухода кожей,293,0,293,1.000000,0.004286
торговых марок,161,1,162,0.993827,0.002355
детей взрослых,159,0,159,1.000000,0.002326


In [249]:
residual_bigram_support = Counter()

for text in residual_text_eval["nameProd_clean"]:
    tokens = [
        token
        for token in text.split()
        if len(token) >= 3
        and not token.isdigit()
    ]

    bigrams = set(zip(tokens, tokens[1:]))

    for a, b in bigrams:
        if a in text_stop_words or b in text_stop_words:
            continue

        residual_bigram_support[f"{a} {b}"] += 1

residual_bigram_support = (
    pd.Series(
        residual_bigram_support,
        name="residual_docs"
    )
    .sort_values(ascending=False)
)

residual_bigram_support.head(50)

мытья посуды                   511
бытовой химии                  501
окрашивания волос              452
товары бытовой                 413
крем краска                    341
интимной гигиены               336
ухода кожей                    293
торговых марок                 161
детей взрослых                 159
гигиеническая моющая           153
зубная паста                   153
косметическая гигиеническая    149
средства моющие                142
посудомоечных машин            141
моющие средства                137
средства косметические         121
средства чистящие              119
полости рта                    106
парфюмерно косметическая       103
детский шампунь                103
товарным знаком                100
жидкое мыло                     97
универсальное моющее            96
салфетки влажные                94
гигиены полости                 90
чистящие средства               89
без аммиака                     87
моющим эффектом                 87
дезинфицирующее моющ

In [250]:
strong_bigrams = [
    "мытья посуды",
    "бытовой химии",
    "окрашивания волос",
    "крем краска",
    "интимной гигиены",
    "зубная паста",
    "детский шампунь",
    "парфюмерно косметическая",
    "жидкое мыло",
    "осветления волос",
    "химической завивки",
]

In [251]:
strong_bigram_mask = (
    residual_text_eval["nameProd_clean"]
    .apply(
        lambda text: any(
            phrase in text
            for phrase in strong_bigrams
        )
    )
)

print("Residual docs:", len(residual_text_eval))
print("Covered by strong bigrams:", strong_bigram_mask.sum())
print("Coverage among residual:", strong_bigram_mask.mean())

Residual docs: 11253
Covered by strong bigrams: 2502
Coverage among residual: 0.2223407091442282


In [252]:
bigram_residual_stats = []

for phrase in strong_bigrams:
    mask = residual_text_eval["nameProd_clean"].str.contains(
        phrase,
        regex=False,
        na=False
    )

    bigram_residual_stats.append({
        "bigram": phrase,
        "docs": int(mask.sum()),
        "share_residual": mask.mean(),
    })

bigram_residual_stats = (
    pd.DataFrame(bigram_residual_stats)
    .sort_values("docs", ascending=False)
)

bigram_residual_stats

,bigram,docs,share_residual
0,мытья посуды,511,0.045410
1,бытовой химии,502,0.044610
2,окрашивания волос,452,0.040167
3,крем краска,341,0.030303
4,интимной гигиены,336,0.029859
5,зубная паста,153,0.013596
6,детский шампунь,103,0.009153
7,парфюмерно косметическая,103,0.009153
8,жидкое мыло,97,0.008620
9,осветления волос,81,0.007198


In [253]:
all_bigram_candidates = (
    bigram_support_clean[
        (bigram_support_clean["35"] >= 30)
        & (bigram_support_clean["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["35", "tg35_precision"],
        ascending=[False, False]
    )
)

all_bigram_candidates.head(30)

tg,35,43,total_docs,tg35_precision,tg35_global_recall
bigram,,,,,
мытья посуды,511,0,511,1.000000,0.007474
бытовой химии,501,11,512,0.978516,0.007328
окрашивания волос,452,0,452,1.000000,0.006611
товары бытовой,413,3,416,0.992788,0.006041
крем краска,341,0,341,1.000000,0.004988
интимной гигиены,336,0,336,1.000000,0.004914
ухода кожей,293,0,293,1.000000,0.004286
торговых марок,161,1,162,0.993827,0.002355
детей взрослых,159,0,159,1.000000,0.002326


In [254]:
candidate_phrases = all_bigram_candidates.index.tolist()

rows = []
covered_docs = set()

for phrase in candidate_phrases:
    mask = residual_text_eval["nameProd_clean"].str.contains(
        phrase,
        regex=False,
        na=False
    )

    new_docs = set(
        residual_text_eval.loc[mask, "rd_documentnumber"]
    ) - covered_docs

    covered_docs.update(new_docs)

    rows.append({
        "bigram": phrase,
        "support": int(mask.sum()),
        "new_docs": len(new_docs),
        "covered_total": len(covered_docs),
        "coverage_all_residual": len(covered_docs) / 11900,
    })

bigram_incremental = pd.DataFrame(rows)

In [255]:
bigram_incremental.head(30)

,bigram,support,new_docs,covered_total,coverage_all_residual
0,мытья посуды,511,511,511,0.042941
1,бытовой химии,502,456,967,0.081261
2,окрашивания волос,452,452,1419,0.119244
3,товары бытовой,413,0,1419,0.119244
4,крем краска,341,181,1600,0.134454
5,интимной гигиены,336,336,1936,0.162689
6,ухода кожей,0,0,1936,0.162689
7,торговых марок,161,136,2072,0.174118
8,детей взрослых,1,1,2073,0.174202
9,гигиеническая моющая,153,129,2202,0.185042


In [256]:
strong_bigram_docs = residual_text_eval.loc[
    strong_bigram_mask,
    "rd_documentnumber"
]

text_covered_residual = tg35_final_uncovered[
    tg35_final_uncovered["rd_documentnumber"]
    .isin(strong_bigram_docs)
]

text_uncovered_residual = tg35_final_uncovered[
    ~tg35_final_uncovered["rd_documentnumber"]
    .isin(strong_bigram_docs)
]

print("Text-covered:", len(text_covered_residual))
print("Still uncovered:", len(text_uncovered_residual))

Text-covered: 2502
Still uncovered: 9398


In [257]:
text_uncovered_residual[
    ["rd_documentnumber", "nameProd"]
].sample(
    50,
    random_state=42
)

,rd_documentnumber,nameProd
4535,RU.43.ОЦ.02.001.R.000036.11.20,"Крем детский от мороза 0+ ""Мапсики""."
5832,RU.62.РЦ.03.015.Е.000946.12.11,Cредство моющее синтетическое порошкообразное ...
5496,RU.54.НС.01.015.Е.000524.08.23,"Универсальное средство для кухни марки ""BOTAVI..."
6418,RU.77.01.34.001.R.000069.01.22,Средство косметическое для детей: Первое очища...
4583,RU.47.01.05.015.Е.000068.12.23,Чистящие средства для текстиля и кожи: Carpet ...
8751,RU.77.99.29.001.R.001094.05.24,Продукция косметическая: Активный ночной отбел...
4896,RU.50.99.05.001.Е.000262.12.13,средство декоративной косметики для макияжа ли...
8678,RU.77.99.29.001.R.000598.03.25,Продукция косметическая: Детский гель для подм...
446,BY.70.06.01.001.R.003261.08.20,HONEY KID 2в1 Средство для купания и шампунь
1308,BY.70.71.01.015.E.000009.01.21,VITEX HOME Чистящий крем для кухни и ванной Ун...


In [258]:
strong_unigrams = (
    tg35_term_candidates
    .head(30)
    .index
    .tolist()
)

strong_bigrams = (
    all_bigram_candidates
    .head(30)
    .index
    .tolist()
)

In [259]:
text_anchor_phrases = strong_unigrams + strong_bigrams

text_anchor_mask = (
    residual_text_eval["nameProd_clean"]
    .apply(
        lambda text: any(
            phrase in text
            for phrase in text_anchor_phrases
        )
    )
)

covered_docs = residual_text_eval.loc[
    text_anchor_mask,
    "rd_documentnumber"
].nunique()

print("Covered residual:", covered_docs)
print("Residual coverage:", covered_docs / 11900)

Covered residual: 9823
Residual coverage: 0.8254621848739496


In [261]:
unigram_mask = (
    residual_text_eval["nameProd_clean"]
    .apply(
        lambda text: any(
            phrase in text
            for phrase in strong_unigrams
        )
    )
)

bigram_mask = (
    residual_text_eval["nameProd_clean"]
    .apply(
        lambda text: any(
            phrase in text
            for phrase in strong_bigrams
        )
    )
)

print("Unigram coverage:", unigram_mask.mean())
print("Bigram coverage:", bigram_mask.mean())
print(
    "Union coverage:",
    (unigram_mask | bigram_mask).mean()
)

Unigram coverage: 0.8542610859326402
Bigram coverage: 0.3218697236292544
Union coverage: 0.8729227761485826


In [262]:
strong_unigram_stats = []

for term in strong_unigrams:
    mask = residual_text_eval["nameProd_clean"].str.contains(
        rf"\b{re.escape(term)}\b",
        regex=True,
        na=False
    )

    strong_unigram_stats.append({
        "term": term,
        "docs": int(mask.sum()),
        "share": mask.mean(),
    })

strong_unigram_stats = (
    pd.DataFrame(strong_unigram_stats)
    .sort_values("docs", ascending=False)
)

strong_unigram_stats

,term,docs,share
0,косметическая,0,0.0
1,марки,0,0.0
2,средства,0,0.0
3,крем,0,0.0
4,волос,0,0.0
5,гель,0,0.0
6,детей,0,0.0
7,серии,0,0.0
8,мытья,0,0.0
9,дезинфицирующее,0,0.0


In [263]:
unigram_incremental = []

covered_docs = set()

for term in strong_unigrams:
    mask = residual_text_eval["nameProd_clean"].str.contains(
        rf"\b{re.escape(term)}\b",
        regex=True,
        na=False
    )

    docs = set(
        residual_text_eval.loc[
            mask,
            "rd_documentnumber"
        ]
    )

    new_docs = docs - covered_docs
    covered_docs.update(docs)

    unigram_incremental.append({
        "term": term,
        "support": len(docs),
        "new_docs": len(new_docs),
        "covered_total": len(covered_docs),
        "coverage_residual": len(covered_docs) / 11900,
    })

unigram_incremental = pd.DataFrame(unigram_incremental)

unigram_incremental.head(30)

,term,support,new_docs,covered_total,coverage_residual
0,косметическая,0,0,0,0.0
1,марки,0,0,0,0.0
2,средства,0,0,0,0.0
3,крем,0,0,0,0.0
4,волос,0,0,0,0.0
5,гель,0,0,0,0.0
6,детей,0,0,0,0.0
7,серии,0,0,0,0.0
8,мытья,0,0,0,0.0
9,дезинфицирующее,0,0,0,0.0


In [264]:
candidate_mask_all = (
    text_eval["nameProd_clean"]
    .apply(
        lambda text: any(
            (
                re.search(
                    rf"\b{re.escape(term)}\b",
                    text
                )
                is not None
            )
            for term in strong_unigrams
        )
        or any(
            phrase in text
            for phrase in strong_bigrams
        )
    )
)

In [265]:
candidate_eval = (
    text_eval.assign(
        candidate_hit=candidate_mask_all
    )
    .groupby("tg")["candidate_hit"]
    .agg(
        docs="sum",
        total="count",
        coverage="mean"
    )
)

candidate_eval

,docs,total,coverage
tg,,,
35,9615,11253,0.854439
43,54,126,0.428571


In [266]:
print(
    residual_text_eval["nameProd_clean"]
    .str.contains("крем", regex=False, na=False)
    .sum()
)

1298


In [267]:
strong_unigram_stats = []

for term in strong_unigrams:
    mask = residual_text_eval["nameProd_clean"].apply(
        lambda text: term in set(text.split())
    )

    strong_unigram_stats.append({
        "term": term,
        "docs": int(mask.sum()),
        "share": mask.mean(),
    })

strong_unigram_stats = (
    pd.DataFrame(strong_unigram_stats)
    .sort_values("docs", ascending=False)
)

strong_unigram_stats

,term,docs,share
0,косметическая,2484,0.220741
1,марки,1895,0.168400
2,средства,1293,0.114903
3,крем,1228,0.109126
4,волос,1205,0.107083
5,гель,1166,0.103617
6,детей,917,0.081489
7,серии,904,0.080334
8,мытья,847,0.075269
9,дезинфицирующее,814,0.072336


In [268]:
unigram_incremental = []

covered_docs = set()

for term in strong_unigrams:
    mask = residual_text_eval["nameProd_clean"].apply(
        lambda text: term in set(text.split())
    )

    docs = set(
        residual_text_eval.loc[
            mask,
            "rd_documentnumber"
        ]
    )

    new_docs = docs - covered_docs
    covered_docs.update(docs)

    unigram_incremental.append({
        "term": term,
        "support": len(docs),
        "new_docs": len(new_docs),
        "covered_total": len(covered_docs),
        "coverage_residual": len(covered_docs) / 11900,
    })

unigram_incremental = pd.DataFrame(unigram_incremental)

unigram_incremental.head(30)

,term,support,new_docs,covered_total,coverage_residual
0,косметическая,2484,2484,2484,0.208739
1,марки,1895,934,3418,0.287227
2,средства,1293,1147,4565,0.383613
3,крем,1228,422,4987,0.419076
4,волос,1205,181,5168,0.434286
5,гель,1166,611,5779,0.485630
6,детей,917,130,5909,0.496555
7,серии,904,248,6157,0.517395
8,мытья,847,484,6641,0.558067
9,дезинфицирующее,814,795,7436,0.624874


In [269]:
candidate_mask_all = (
    text_eval["nameProd_clean"].apply(
        lambda text: (
            any(term in set(text.split()) for term in strong_unigrams)
            or any(phrase in text for phrase in strong_bigrams)
        )
    )
)

In [270]:
candidate_eval = (
    text_eval.assign(
        candidate_hit=candidate_mask_all
    )
    .groupby("tg")["candidate_hit"]
    .agg(
        docs="sum",
        total="count",
        coverage="mean"
    )
)

candidate_eval

,docs,total,coverage
tg,,,
35,9615,11253,0.854439
43,54,126,0.428571


In [271]:
text_docs = (
    base_df[
        ["rd_documentnumber", "tg", "nameProd"]
    ]
    .explode("tg", ignore_index=True)
    .copy()
)

text_docs["tg"] = text_docs["tg"].astype("string")

text_docs["nameProd_clean"] = (
    text_docs["nameProd"]
    .apply(normalize_text)
)

text_docs = text_docs[
    text_docs["nameProd_clean"].ne("")
    & text_docs["tg"].isin(["35", "43"])
].copy()

print(text_docs["tg"].value_counts())
print(text_docs["rd_documentnumber"].nunique())

tg
35    11253
43      126
Name: count, dtype: int64[pyarrow]
11362


In [273]:
import numpy as np

doc_ids = text_docs["rd_documentnumber"].astype(str).unique().to_numpy()

train_ids, valid_ids = train_test_split(
    doc_ids,
    test_size=0.25,
    random_state=42
)

In [274]:
train_text = text_docs[
    text_docs["rd_documentnumber"].isin(train_ids)
].copy()

valid_text = text_docs[
    text_docs["rd_documentnumber"].isin(valid_ids)
].copy()

print("Train:", train_text.shape)
print("Valid:", valid_text.shape)
print(
    "Train docs:",
    train_text["rd_documentnumber"].nunique()
)
print(
    "Valid docs:",
    valid_text["rd_documentnumber"].nunique()
)

Train: (8532, 4)
Valid: (2847, 4)
Train docs: 8521
Valid docs: 2841


In [275]:
doc_labels = (
    text_docs[
        ["rd_documentnumber", "tg"]
    ]
    .drop_duplicates("rd_documentnumber")
)

print(doc_labels["tg"].value_counts())

tg
35    11253
43      109
Name: count, dtype: int64[pyarrow]


In [276]:
doc_labels.groupby("rd_documentnumber")["tg"].nunique().value_counts()

tg
1    11362
Name: count, dtype: int64

In [277]:
train_term_docs = []

for tg, rd_number, text in zip(
    train_text["tg"],
    train_text["rd_documentnumber"],
    train_text["nameProd_clean"]
):
    tokens = {
        token
        for token in text.split()
        if token not in text_stop_words
        and len(token) >= 3
        and not token.isdigit()
    }

    for term in tokens:
        train_term_docs.append(
            (term, tg, rd_number)
        )

train_term_docs = (
    pd.DataFrame(
        train_term_docs,
        columns=["term", "tg", "rd_number"]
    )
    .drop_duplicates()
)

In [278]:
train_term_support = (
    train_term_docs
    .groupby(["term", "tg"])["rd_number"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["35", "43"], fill_value=0)
)

train_term_support["total_docs"] = (
    train_term_support["35"]
    + train_term_support["43"]
)

train_term_support["tg35_precision"] = (
    train_term_support["35"]
    / train_term_support["total_docs"]
)

train_term_support["tg35_recall"] = (
    train_term_support["35"]
    / train_text.loc[
        train_text["tg"] == "35",
        "rd_documentnumber"
    ].nunique()
)

In [279]:
train_candidates = (
    train_term_support[
        (train_term_support["35"] >= 50)
        & (train_term_support["tg35_precision"] >= 0.90)
    ]
    .sort_values(
        ["35", "tg35_precision"],
        ascending=[False, False]
    )
)

train_candidates.head(30)

tg,35,43,total_docs,tg35_precision,tg35_recall
term,,,,,
косметическая,1849,2,1851,0.998920,0.219076
средства,978,15,993,0.984894,0.115877
крем,923,3,926,0.996760,0.109360
волос,894,1,895,0.998883,0.105924
гель,889,0,889,1.000000,0.105332
детей,696,2,698,0.997135,0.082464
мытья,639,0,639,1.000000,0.075711
моющее,631,1,632,0.998418,0.074763
дезинфицирующее,625,0,625,1.000000,0.074052


In [280]:
valid_terms = train_candidates.head(30).index.tolist()

In [281]:
valid_hit = valid_text["nameProd_clean"].apply(
    lambda text: any(
        term in set(text.split())
        for term in valid_terms
    )
)

valid_eval = (
    valid_text.assign(hit=valid_hit)
    .groupby("tg")["hit"]
    .agg(
        docs="sum",
        total="count",
        coverage="mean"
    )
)

valid_eval

,docs,total,coverage
tg,,,
35,2306,2813,0.819765
43,14,34,0.411765


In [282]:
valid_text.loc[
    (valid_text["tg"] == "43")
    & valid_hit,
    ["rd_documentnumber", "nameProd"]
].head(30)

,rd_documentnumber,nameProd
2650,RU.01.РА.02.001.R.002024.10.22,Продукция косметическая для детей: Крем под по...
4271,RU.30.АЦ.02.015.Е.000522.10.25,Стеклоомывающая жидкость с маркировкой UnitCle...
4425,RU.33.ВЛ.04.015.Е.000174.03.12,Товары бытовой химии (автохимия) (далее согла...
5298,RU.54.НС.01.015.Е.000002.01.21,Средства бытовой химии в аэрозольной упаковке:...
5858,RU.61.РЦ.10.015.Е.000065.10.18,Средство бытовой химии по уходу за автомобилям...
6392,RU.67.СО.01.015.Е.006114.12.11,Средства автокосметики в аэрозольной упаковке:...
7934,RU.77.01.34.015.Е.001591.04.14,Средство бытовой химии: Размораживатель замков...
8044,RU.77.01.34.015.Е.002067.07.17,"Очиститель тормозов с маркировкой «Wurth», арт..."
8046,RU.77.01.34.015.Е.002071.08.19,Средства по уходу за автомобилями в аэрозольно...
8315,RU.77.01.34.015.Е.003439.12.17,Средства по уходу за автомобилями: разморажива...


In [283]:
valid_anchor = valid_text[
    ["rd_documentnumber", "tg"]
].drop_duplicates()

valid_anchor = valid_anchor.merge(
    base_df[
        ["rd_documentnumber", "tnved_4"]
    ],
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
              rd_documentnumber
RU.01.РА.02.001.R.002024.10.22
RU.54.НС.01.015.Е.000002.01.21
RU.66.01.40.015.Е.000160.11.21
RU.76.01.07.015.Е.000007.03.17
RU.77.01.34.015.Е.010593.12.12 ...

In [286]:
print(
    "Duplicate docs in valid_text:",
    valid_text["rd_documentnumber"].duplicated().sum()
)

print(
    "Duplicate docs in train_text:",
    train_text["rd_documentnumber"].duplicated().sum()
)

Duplicate docs in valid_text: 6
Duplicate docs in train_text: 11


In [287]:
valid_text[
    valid_text["rd_documentnumber"].duplicated(keep=False)
].sort_values("rd_documentnumber").head(30)

,rd_documentnumber,tg,nameProd,nameProd_clean
2649,RU.01.РА.02.001.R.002024.10.22,35,Продукция косметическая для детей: Крем под по...,продукция косметическая для детей крем под под...
2650,RU.01.РА.02.001.R.002024.10.22,43,Продукция косметическая для детей: Крем под по...,продукция косметическая для детей крем под под...
5297,RU.54.НС.01.015.Е.000002.01.21,35,Средства бытовой химии в аэрозольной упаковке:...,средства бытовой химии в аэрозольной упаковке ...
5298,RU.54.НС.01.015.Е.000002.01.21,43,Средства бытовой химии в аэрозольной упаковке:...,средства бытовой химии в аэрозольной упаковке ...
6207,RU.66.01.40.015.Е.000160.11.21,35,Очиститель системы охлаждения,очиститель системы охлаждения
6208,RU.66.01.40.015.Е.000160.11.21,43,Очиститель системы охлаждения,очиститель системы охлаждения
6482,RU.76.01.07.015.Е.000007.03.17,35,"Средство для очистки, разморозки стекол и замк...",средство для очистки разморозки стекол и замко...
6483,RU.76.01.07.015.Е.000007.03.17,43,"Средство для очистки, разморозки стекол и замк...",средство для очистки разморозки стекол и замко...
8423,RU.77.01.34.015.Е.010593.12.12,35,Жидкости стеклоочищающие комплексного действия...,жидкости стеклоочищающие комплексного действия...
8424,RU.77.01.34.015.Е.010593.12.12,43,Жидкости стеклоочищающие комплексного действия...,жидкости стеклоочищающие комплексного действия...


In [288]:
valid_anchor = valid_text[
    ["rd_documentnumber", "tg"]
].drop_duplicates()

In [289]:
valid_anchor = (
    valid_text[
        ["rd_documentnumber", "tg"]
    ]
    .drop_duplicates("rd_documentnumber")
    .copy()
)

In [290]:
valid_anchor = valid_anchor.merge(
    base_df[
        ["rd_documentnumber", "tnved_4"]
    ],
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

In [303]:
valid_anchor = (
    text_docs[
        text_docs["rd_documentnumber"].isin(valid_ids)
    ][
        ["rd_documentnumber", "tg"]
    ]
    .drop_duplicates("rd_documentnumber")
    .copy()
)

valid_anchor = valid_anchor.merge(
    base_df[
        ["rd_documentnumber", "tnved_4"]
    ],
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

print(valid_anchor.shape)
print(valid_anchor["rd_documentnumber"].nunique())
print(valid_anchor["tg"].value_counts())

(2841, 3)
2841
tg
35    2813
43      28
Name: count, dtype: int64[pyarrow]


In [293]:
prefix_to_tg = {
    "3303": "4",
    "3304": "35",
    "3305": "35",
    "3401": "35",
    "2710": "43",
    "3403": "43",
    "3306": "35",
    "3307": "35",
    "3402": "35",
    "3808": "35",
}

In [294]:
def get_tnved_anchor_tg(codes):
    for code in codes:
        if code in prefix_to_tg:
            return prefix_to_tg[code]
    return None

In [304]:
valid_anchor["tnved_anchor_tg"] = (
    valid_anchor["tnved_4"]
    .apply(get_tnved_anchor_tg)
)

In [305]:
valid35 = valid_anchor[
    valid_anchor["tg"] == "35"
].copy()

valid35["tnved_covered"] = (
    valid35["tnved_anchor_tg"] == "35"
)

valid35["residual"] = (
    ~valid35["tnved_covered"]
)

print("Validation TG35:", len(valid35))
print(
    "TNVED covered:",
    valid35["tnved_covered"].sum()
)
print(
    "Residual:",
    valid35["residual"].sum()
)

print(
    "TNVED coverage:",
    valid35["tnved_covered"].mean()
)

Validation TG35: 2813
TNVED covered: 0
Residual: 2813
TNVED coverage: 0.0


In [306]:
valid35_residual = valid35[
    valid35["residual"]
].merge(
    valid_text[
        ["rd_documentnumber", "nameProd_clean"]
    ].drop_duplicates("rd_documentnumber"),
    on="rd_documentnumber",
    how="left",
    validate="one_to_one"
)

In [307]:
residual_valid_hit = (
    valid35_residual["nameProd_clean"]
    .fillna("")
    .apply(
        lambda text: any(
            term in set(text.split())
            for term in valid_terms
        )
    )
    .astype(bool)
)

In [308]:
print(
    "Validation TG35 residual:",
    len(valid35_residual)
)

print(
    "Text covered:",
    int(residual_valid_hit.sum())
)

print(
    "Text coverage:",
    residual_valid_hit.mean()
)

Validation TG35 residual: 2813
Text covered: 2306
Text coverage: 0.8197653750444366


In [309]:
print(
    "Document overlap:",
    len(set(train_ids) & set(valid_ids))
)

Document overlap: 0


In [310]:
print(
    "Train TG distribution:"
)

print(
    train_text["tg"].value_counts()
)

print(
    "\nValid TG distribution:"
)

print(
    valid_text["tg"].value_counts()
)

Train TG distribution:
tg
35    8440
43      92
Name: count, dtype: int64[pyarrow]

Valid TG distribution:
tg
35    2813
43      34
Name: count, dtype: int64[pyarrow]


In [311]:
print(valid_anchor.columns)
print(valid_anchor["tnved_4"].head(10).tolist())

Index(['rd_documentnumber', 'tg', 'tnved_4', 'tnved_anchor_tg'], dtype='str')
[[], [], [], [], [], [], [], [], [], []]


In [312]:
print(valid_anchor["tnved_4"].apply(type).value_counts())

tnved_4
<class 'list'>    2841
Name: count, dtype: int64


In [313]:
for codes in valid_anchor["tnved_4"].head(20):
    print(codes, "->", get_tnved_anchor_tg(codes))

[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None
[] -> None


In [317]:
print(
    base_df.loc[
        base_df["rd_documentnumber"].isin(valid_ids),
        "tnved_list"
    ].apply(len).describe()
)

count    2841.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: tnved_list, dtype: float64


In [318]:
print(
    base_df.loc[
        base_df["rd_documentnumber"].isin(valid_ids),
        "tnved_list"
    ].apply(len).value_counts().sort_index()
)

tnved_list
0    2841
Name: count, dtype: int64


In [319]:
valid_base = base_df[
    base_df["rd_documentnumber"].isin(valid_ids)
].copy()

valid_base["tnved_present"] = (
    valid_base["tnved_list"].apply(len) > 0
)

print(
    valid_base.groupby(
        valid_base["tg"].apply(lambda x: 35 in x)
    )["tnved_present"].mean()
)

tg
False    0.0
Name: tnved_present, dtype: float64


In [320]:
valid_base35 = valid_base[
    valid_base["tg"].apply(lambda x: 35 in x)
]

print(
    "Validation TG35:",
    len(valid_base35)
)

print(
    "TG35 with TNVED:",
    valid_base35["tnved_list"].apply(len).gt(0).sum()
)

print(
    "TG35 TNVED coverage:",
    valid_base35["tnved_list"].apply(len).gt(0).mean()
)

Validation TG35: 0
TG35 with TNVED: 0
TG35 TNVED coverage: nan


In [321]:
valid_base35[
    valid_base35["tnved_list"].apply(len).gt(0)
][
    [
        "rd_documentnumber",
        "tnved_list",
        "tnved_4"
    ]
].head(20)

,rd_documentnumber,tnved_list,tnved_4
